In [1]:
from dotenv import load_dotenv
import asyncio
import json

load_dotenv()

True

In [2]:
# ── Test Utils ─────────────────────────────────────────────────────

async def run_with_stream(agent, prompt, formatter: str = "json", *, print_chunks: bool = True):
    """Run agent.run_stream with auto-managed queue and printer."""
    queue: asyncio.Queue[str | None] = asyncio.Queue()

    async def _drain():
        while True:
            chunk = await queue.get()
            if chunk is None:
                break
            if print_chunks:
                print(chunk, flush=True)
            queue.task_done()

    printer = asyncio.create_task(_drain())
    result = await agent.run_stream(prompt, queue, formatter)
    await queue.put(None)
    await printer
    return result


class DemoHandle:
    """Handle for an in-flight streaming run (abort/steer testing)."""

    def __init__(self, agent, runner, queue, collector):
        self.agent = agent
        self._runner = runner
        self._queue = queue
        self._collector = collector

    async def _cleanup(self):
        await self._queue.put(None)
        await self._collector

    async def abort(self):
        result = await self.agent.abort()
        if not self._runner.done():
            await self._runner
        await self._cleanup()
        _print_result(result)
        return result

    async def steer(self, instruction: str):
        result = await self.agent.steer(
            instruction,
            queue=self._queue,
            stream_formatter="json",
        )
        if not self._runner.done():
            await self._runner
        await self._cleanup()
        _print_result(result)
        return result


def _print_result(result) -> None:
    print(f"stop_reason={result.stop_reason}")
    print(f"was_aborted={getattr(result, 'was_aborted', None)}")
    print(f"abort_phase={getattr(result, 'abort_phase', None)}")
    print(f"total_steps={result.total_steps}")
    print(f"final_answer={result.final_answer!r}")


async def start_demo(agent, prompt: str, formatter: str = "json") -> DemoHandle:
    """Start a streaming run and return a handle for abort/steer."""
    queue: asyncio.Queue[str | None] = asyncio.Queue()
    chunks: list[str] = []

    async def _collect():
        while True:
            chunk = await queue.get()
            if chunk is None:
                break
            chunks.append(chunk)
            print(chunk, flush=True)
            queue.task_done()

    collector = asyncio.create_task(_collect())
    runner = asyncio.create_task(agent.run_stream(prompt, queue, formatter))
    return DemoHandle(agent, runner, queue, collector)

# Tool Schema Tests

In [2]:
from typing import Any

from agent_base.tools import ToolRegistry, tool


@tool
def get_weather(location: str, unit: str = "celsius") -> str:
    """Get the current weather in a given location.
    
    This function retrieves weather information for the specified location
    and returns it in the requested temperature unit.
    
    Args:
        location: The city and state, e.g. "San Francisco, CA"
        unit: The unit of temperature, either "celsius" or "fahrenheit"
    
    Returns:
        String describing the current weather
    """
    # Mock implementation
    return f"The weather in {location} is 22°{unit[0].upper()}"

@tool
def plan_travel_itinerary(
    travelers: list[dict[str, str]],
    destination: str,
    start_date: str,
    end_date: str,
    preferences: dict[str, Any] | None = None,
) -> str:
    """Plan a multi-day travel itinerary.
    
    This tool accepts structured traveler information along with
    trip preferences and returns a concise summary. It showcases
    nested objects and optional parameters in the schema.
    
    Args:
        travelers: List of traveler profiles including name and role
        destination: Target city or country for the itinerary
        start_date: Trip start in ISO format (YYYY-MM-DD)
        end_date: Trip end in ISO format (YYYY-MM-DD)
        preferences: Optional mapping of preference category to value
    
    Returns:
        String summary describing the generated itinerary
    """
    traveler_list = ", ".join(
        f"{traveler['name']} ({traveler.get('role', 'guest')})"
        for traveler in travelers
    )
    pref_summary = (
        ", ".join(f"{key}={value}" for key, value in preferences.items())
        if preferences
        else "standard preferences"
    )
    return (
        f"Trip to {destination} from {start_date} to {end_date} for "
        f"{len(travelers)} traveler(s): {traveler_list}. Preferences: {pref_summary}."
    )


registry = ToolRegistry()
registry.register_tools([get_weather, plan_travel_itinerary])
print("Registered tools:", [schema.name for schema in registry.get_schemas()])

Registered tools: ['get_weather', 'plan_travel_itinerary']


## Export as Anthropic Format

In [3]:
import json
from dataclasses import asdict

anthropic_tool_schemas = registry.get_schemas()
print(json.dumps([asdict(s) for s in anthropic_tool_schemas], indent=2))

[
  {
    "name": "get_weather",
    "description": "Get the current weather in a given location.\n\nThis function retrieves weather information for the specified location\nand returns it in the requested temperature unit.",
    "input_schema": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string",
          "description": "The city and state, e.g. \"San Francisco, CA\""
        },
        "unit": {
          "type": "string",
          "description": "The unit of temperature, either \"celsius\" or \"fahrenheit\""
        }
      },
      "required": [
        "location"
      ]
    }
  },
  {
    "name": "plan_travel_itinerary",
    "description": "Plan a multi-day travel itinerary.\n\nThis tool accepts structured traveler information along with\ntrip preferences and returns a concise summary. It showcases\nnested objects and optional parameters in the schema.",
    "input_schema": {
      "type": "object",
      "properties": {
        "trav

# Basic Agent with Tools

In [2]:
from agent_base.storage import create_adapters

In [3]:
# Create all three storage adapters with filesystem backend
# config_adapter, conversation_adapter, run_adapter = create_adapters(
#     "filesystem",
#     base_path="./test_data"
# )

In [4]:
# Alternatively, use filesystem adapters for cloud persistence
import os
config_adapter, conversation_adapter, run_adapter = create_adapters(
    "postgres",
    connection_string=os.getenv("DATABASE_URL")
)

In [5]:
import asyncio

from agent_base.providers.anthropic import AnthropicAgent
from agent_base.tools import tool


@tool
def add(a: int, b: int) -> str:
    """Add two numbers together.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        The sum as a string
    """
    return str(a + b)

@tool
def multiply(a: int, b: int) -> str:
    """Multiply two numbers together.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        The product as a string
    """
    return str(a * b)

SAMPLE_TOOLS = [add, multiply]

agent = AnthropicAgent(
    system_prompt="You are a helpful assistant that should help the user with their questions.",
    model="claude-sonnet-4-5",
    tools=SAMPLE_TOOLS,
    config_adapter=config_adapter,
    conversation_adapter=conversation_adapter,
    run_adapter=run_adapter,
)

In [6]:
await agent.initialize()
agent_uuid = agent.agent_uuid
print(agent_uuid)

c77909a9-b93d-4538-b024-6428027afd08


In [7]:
print(agent)

In [8]:
prompt = "What is 5 + (10 * 2)?"

## Running w/o queue

In [9]:
res_wo_queue = await agent.run(prompt=prompt)

2026-03-06T04:42:33.266772Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-06T04:42:36.097928Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-06T04:42:38.131668Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-06T04:42:38.509945Z [info     ] Created PostgreSQL connection pool [agent_base.storage.adapters.postgres] max_size=10


In [10]:
print(res_wo_queue)

AgentResult(final_message=Message(id='012065db-8cb3-43dd-8aa3-b932ea8c700f', role=<Role.ASSISTANT: 'assistant'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text="The answer is **25**.\n\nHere's how it breaks down:\n- First: 10 * 2 = 20\n- Then: 5 + 20 = 25")], stop_reason='end_turn', usage=Usage(input_tokens=890, output_tokens=45, cache_write_tokens=0, cache_read_tokens=0, thinking_tokens=None, raw_usage={'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 890, 'output_tokens': 45, 'service_tier': 'standard'}), provider='anthropic', model='claude-sonnet-4-5-20250929', usage_kwargs={}), final_answer="The answer is **25**.\n\nHere's how it breaks down:\n- First: 10 * 2 = 20\n- Then: 5 + 20 = 25", conversation_history=[Message(id='1f7b0464-0272-4a75-9767-13e2baa0ab2e', role=<Role.USER: 'user'>, content=[T

In [11]:
print(res_wo_queue.final_message.content[0].text)

The answer is **25**.

Here's how it breaks down:
- First: 10 * 2 = 20
- Then: 5 + 20 = 25


In [12]:
print(res_wo_queue.final_answer)

The answer is **25**.

Here's how it breaks down:
- First: 10 * 2 = 20
- Then: 5 + 20 = 25


## Running with Queue

### JSON Formatter (Default)

The JSON formatter is now the default and recommended formatter. It emits self-contained JSON envelopes with message types like `text`, `thinking`, `tool_call`, `tool_result`, etc. Each envelope includes the agent UUID and a `final` flag for stream completion.

In [13]:
result = await run_with_stream(agent, prompt)

2026-03-06T04:42:50.445888Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":"I"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" need to calculate 5 + ("}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":"10 * 2)."}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" Following the order of operations, I should"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" first multiply 10 * 2"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":", then add 5 to the"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" result."}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":true,"delta":""}
{"type":"tool_call","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":true,"delta":"{\"a\": 10, \"b\": 2}","id":"toolu_01JPcsEhpPFjnsJ4SE61JUfb","name":"multiply"}


2026-03-06T04:42:53.395403Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"tool_call","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":true,"delta":"{\"a\": 5, \"b\": 20}","id":"toolu_0161L15uYU9hWDcmUyBBtfus","name":"add"}


2026-03-06T04:42:55.798579Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":"The"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" answer is **25**.\n\nHere's"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" how it breaks down:\n- First"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":": 10 * 2 "}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":"= 20\n- Then: "}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":"5 + 20 = "}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":"25"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":true,"delta":""}


In [14]:
print(result.final_answer)

The answer is **25**.

Here's how it breaks down:
- First: 10 * 2 = 20
- Then: 5 + 20 = 25


#### Agent Resume

In [15]:
import nest_asyncio
nest_asyncio.apply()

In [16]:
agent2 = AnthropicAgent(
    agent_uuid=agent_uuid,
    tools=SAMPLE_TOOLS,
    config_adapter=config_adapter,
    conversation_adapter=conversation_adapter,
    run_adapter=run_adapter,
)
print(agent2.agent_uuid)

c77909a9-b93d-4538-b024-6428027afd08


In [17]:
print(agent2)
prompt2 = "What did you do before? Explain in detail."

In [18]:
result2 = await run_with_stream(agent2, prompt2)

2026-03-06T04:43:31.324290Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":"Before,"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" I calculate"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":"d the"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" expression"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" 5 + (10 *"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" 2) for"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" you."}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" Here"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":"'s a"}
{"type":"text","agent":"c77909a9-b93d-4538-b024-6428027afd08","final":false,"delta":" detailed explanation of what"}
{"type":"text","agent":"c

In [19]:
print(result2.final_answer)

Before, I calculated the expression 5 + (10 * 2) for you. Here's a detailed explanation of what I did:

**Step 1: Understanding the Problem**
You asked me to calculate 5 + (10 * 2). This is a mathematical expression that requires following the order of operations (PEMDAS/BODMAS), where multiplication must be performed before addition.

**Step 2: First Calculation - Multiplication**
I called the `multiply` function with:
- Parameter a = 10
- Parameter b = 2
- Result: 20

This solved the part inside the parentheses: (10 * 2) = 20

**Step 3: Second Calculation - Addition**
I then called the `add` function with:
- Parameter a = 5
- Parameter b = 20 (the result from the multiplication)
- Result: 25

This completed the calculation: 5 + 20 = 25

**Step 4: Providing the Answer**
I reported back that the final answer is **25** and showed you the breakdown of how I arrived at that answer.

The key point is that I had to perform the calculations in two separate steps because the tools I have acce

# Agent With Server Tools

In [57]:
from datetime import datetime
from agent_base.providers.anthropic import AnthropicAgent, AnthropicLLMConfig

today_date = datetime.now().strftime("%Y-%m-%d")

SYSTEM_PROMPT = f"""Today's date is {today_date}.
You are a helpful financial and business analyst assistant. 
You are tasked with answering questions about Paytm, an Indian fintech company. 
Provide clear, concise, and accurate answers based on the question asked.
If you don't have enough information to answer accurately, say so."""

agent = AnthropicAgent(
    system_prompt=SYSTEM_PROMPT,
    model="claude-sonnet-4-5",
    config=AnthropicLLMConfig(
        server_tools=[{
            "type": "code_execution_20250825",
            "name": "code_execution"
        },
        {
            "type": "web_search_20250305",
            "name": "web_search",
            "max_uses": 50
        },
        {
            "type": "web_fetch_20250910",
            "name": "web_fetch",
            "max_uses": 50
        }],
        beta_headers=["code-execution-2025-08-25", "web-fetch-2025-09-10", "files-api-2025-04-14"],
    ),
    stream_meta_history_and_tool_results=False,
)

In [58]:
await agent.initialize()
agent_uuid = agent.agent_uuid
print(agent_uuid)

36286532-9866-4018-8f49-51787666071d


In [59]:
print(agent)

In [60]:
from agent_base.core.types import ContentBlockType, TextContent

def collate_final_text(message) -> str:
    """
    Collate all text blocks after the last server tool use/result block.
    
    Args:
        message: The Message (e.g., result.final_message)
        
    Returns:
        Concatenated text from all text blocks after the last tool block
    """
    
    if not message or not message.content:
            return ""

    TOOL_BLOCK_TYPES = {
        ContentBlockType.SERVER_TOOL_USE,
        ContentBlockType.SERVER_TOOL_RESULT,
        ContentBlockType.TOOL_USE,
        ContentBlockType.TOOL_RESULT,
    }

    start_index = 0
    # Find the index of the last tool_use block, if any
    for i, block in enumerate(message.content):
        if block.content_block_type in TOOL_BLOCK_TYPES:
            start_index = i + 1
    
    full_text = []
    for block in message.content[start_index:]:
        if isinstance(block, TextContent):
            full_text.append(block.text)
    
    return "".join(full_text)

In [61]:
# prompt = "Summarize Paytm's latest quarterly (Q2 FY 2026) performance and key drivers"
# prompt = "Compare Paytm's latest quarterly (Q2 FY 2026) performance with Q2 FY 2025 performance"
# prompt = "What is the base, adjusted, normalized ebitda for paytm in FY25? Mention the formula used for calculation."
prompt = "What is the whether today in Bangalore?"

In [62]:
import asyncio
queue: asyncio.Queue[str | None] = asyncio.Queue()

In [63]:
import nest_asyncio
nest_asyncio.apply()

In [64]:
res_wo_queue = await agent.run_stream(
    prompt=prompt,
    queue=queue,
)

2026-03-05T14:55:05.349901Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


In [65]:
result = await run_with_stream(agent, prompt)

2026-03-05T14:55:12.515109Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"36286532-9866-4018-8f49-51787666071d","final":false,"delta":"Based"}
{"type":"text","agent":"36286532-9866-4018-8f49-51787666071d","final":false,"delta":" on the search"}
{"type":"text","agent":"36286532-9866-4018-8f49-51787666071d","final":false,"delta":" I"}
{"type":"text","agent":"36286532-9866-4018-8f49-51787666071d","final":false,"delta":" just"}
{"type":"text","agent":"36286532-9866-4018-8f49-51787666071d","final":false,"delta":" performed, here"}
{"type":"text","agent":"36286532-9866-4018-8f49-51787666071d","final":false,"delta":"'s today"}
{"type":"text","agent":"36286532-9866-4018-8f49-51787666071d","final":false,"delta":"'s weather in Bangalore:"}
{"type":"text","agent":"36286532-9866-4018-8f49-51787666071d","final":false,"delta":"\n\n"}
{"type":"text","agent":"36286532-9866-4018-8f49-51787666071d","final":true,"delta":""}
{"type":"text","agent":"36286532-9866-4018-8f49-51787666071d","final":false,"delta":"Current weather conditions show it"}
{"type":"

In [66]:
print(result.final_answer)

Based on the search I just performed, here's today's weather in Bangalore:

Current weather conditions show it's sunny with a temperature of 31°C (approximately 88°F), humidity at 29%, and winds at 21.6 km/h. The forecast indicates clear skies with a low temperature around 71°F (approximately 22°C).

Air quality has reached a high level of pollution and is unhealthy for sensitive groups, with advice to reduce time spent outside if experiencing symptoms.

It's a warm, sunny day in Bangalore with relatively low humidity and good visibility!


In [67]:
res_wo_queue.final_message

Message(id='ee8002fe-de8a-4038-a060-f2ad94407d4d', role=<Role.ASSISTANT: 'assistant'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text="I'll search for today's weather in Bangalore for you."), ServerToolUseContent(content_block_type=<ContentBlockType.SERVER_TOOL_USE: 'server_tool_use'>, kwargs={'caller': {'type': 'direct'}}, tool_name='web_search', tool_id='srvtoolu_01TSgE9nBUXmHFHTo4BB8bYM', tool_input={'query': 'Bangalore weather today'}), ServerToolResultContent(content_block_type=<ContentBlockType.SERVER_TOOL_RESULT: 'server_tool_result'>, kwargs={'caller': {'type': 'direct'}}, tool_name='web_search_tool_result', tool_id='srvtoolu_01TSgE9nBUXmHFHTo4BB8bYM', tool_result=[{'encrypted_content': 'ErcLCioIDRgCIiQxMDM4NWM5Ny0wZjFhLTQwMWEtYTBlOS00YjhmMzI2ODFhNTQSDCIYvnDlGkfF6c+aFhoMHBxG+8cf8DuElaBcIjC4gEtuo81ZDvHlwEtFhUxZRvpGrp1x+40+Bfj2LmTEG7dnPYrft7sftInvsmc5FS4qugqp7Crq625gt2lfrkVDLrULGKol6rCJ0YxKS/u3yY0rZrKry/4SSvoYvhPh9xuMPuRgtnCoGPFwiEVlh/MQB

In [68]:
print(res_wo_queue.final_answer)

I'll search for today's weather in Bangalore for you.Based on the search results, here's today's weather in Bangalore:

Current weather conditions show it's sunny with a temperature of 31°C (approximately 88°F), humidity at 29%, and winds at 21.6 km/h. The forecast indicates clear skies with a low temperature around 71°F (approximately 22°C).

Air quality has reached a high level of pollution and is unhealthy for sensitive groups, with advice to reduce time spent outside if experiencing symptoms.

It's a warm, sunny day in Bangalore with relatively low humidity and good visibility!


In [69]:
final_text = collate_final_text(res_wo_queue.final_message)
print(final_text)

Based on the search results, here's today's weather in Bangalore:

Current weather conditions show it's sunny with a temperature of 31°C (approximately 88°F), humidity at 29%, and winds at 21.6 km/h. The forecast indicates clear skies with a low temperature around 71°F (approximately 22°C).

Air quality has reached a high level of pollution and is unhealthy for sensitive groups, with advice to reduce time spent outside if experiencing symptoms.

It's a warm, sunny day in Bangalore with relatively low humidity and good visibility!


# Agent with answer check

In [70]:
import re

def validate_answer(answer: str) -> tuple[bool, str]:
    """
    Validate the answer based on the question asked.
    """
    text = answer
    
    if not text or not text.strip():
        return False, "Input text is empty"
    
    # Try to extract JSON from the text
    json_candidates = []
    
    # Method 1: Look for JSON in markdown code blocks
    code_block_pattern = r'```(?:json)?\s*(\{.*?\})\s*```'
    code_blocks = re.findall(code_block_pattern, text, re.DOTALL)
    json_candidates.extend(code_blocks)
    
    # Method 2: Look for JSON objects in the text (handle both {} and nested structures)
    json_object_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    json_objects = re.findall(json_object_pattern, text, re.DOTALL)
    json_candidates.extend(json_objects)
    
    # Method 3: Try the entire text as JSON
    json_candidates.append(text.strip())
    
    # Try to parse each candidate
    for candidate in json_candidates:
        candidate = candidate.strip()
        if not candidate:
            continue
            
        try:
            # Attempt to parse JSON
            parsed_json = json.loads(candidate)
            
            # Validate it's a dictionary
            if not isinstance(parsed_json, dict):
                continue
            
            # Check for required fields
            if 'name' not in parsed_json:
                continue
            
            if 'description' not in parsed_json:
                continue
            
            # Validate field types (should be strings or at least present)
            name_value = parsed_json['name']
            description_value = parsed_json['description']
            
            # Check if values are None
            if name_value is None or description_value is None:
                continue
            
            # Successfully found and validated JSON
            return True, json.dumps(parsed_json, ensure_ascii=False)
            
        except json.JSONDecodeError:
            # This candidate wasn't valid JSON, try next one
            continue
        except Exception as e:
            # Unexpected error, continue to next candidate
            continue
    
    # If we get here, no valid JSON was found
    return False, "No valid JSON with 'name' and 'description' fields found in text"

In [71]:
import asyncio

from agent_base.providers.anthropic import AnthropicAgent, AnthropicLLMConfig

agent = AnthropicAgent(
    system_prompt="""
    You are a helpful assistant that should help the user with their questions.
    You should always return the answer in JSON format only with the following fields:
    - name
    - description
    """,
    model="claude-sonnet-4-5",
    final_answer_check=validate_answer,
    config=AnthropicLLMConfig(
        server_tools=[
            {
                "type": "web_search_20250305",
                "name": "web_search",
                "max_uses": 50
            },
            {
                "type": "web_fetch_20250910",
                "name": "web_fetch",
                "max_uses": 50
            }
        ],
        beta_headers=["web-fetch-2025-09-10"],
    ),
)

In [72]:
print(agent)
prompt = """
Search various products on Amazon in the category of gaming laptops.
Return any 5 results in JSON format with the following fields:
- name
- description
"""

In [73]:
result = await run_with_stream(agent, prompt)

2026-03-05T14:56:23.974611Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"ef4dbdf5-33c1-4b6f-890c-7fe7600b36ee","final":false,"delta":"I'll search for gaming laptops on"}
{"type":"text","agent":"ef4dbdf5-33c1-4b6f-890c-7fe7600b36ee","final":false,"delta":" Amazon and"}
{"type":"text","agent":"ef4dbdf5-33c1-4b6f-890c-7fe7600b36ee","final":false,"delta":" provide"}
{"type":"text","agent":"ef4dbdf5-33c1-4b6f-890c-7fe7600b36ee","final":false,"delta":" you with 5 results."}
{"type":"text","agent":"ef4dbdf5-33c1-4b6f-890c-7fe7600b36ee","final":true,"delta":""}
{"type":"server_tool_call","agent":"ef4dbdf5-33c1-4b6f-890c-7fe7600b36ee","final":true,"delta":"{\"query\": \"Amazon gaming laptops\"}","id":"srvtoolu_018sufNGFQcVzSxc6LsEu3KC","name":"web_search"}
{"type":"text","agent":"ef4dbdf5-33c1-4b6f-890c-7fe7600b36ee","final":false,"delta":"Let"}
{"type":"text","agent":"ef4dbdf5-33c1-4b6f-890c-7fe7600b36ee","final":false,"delta":" me fetch"}
{"type":"text","agent":"ef4dbdf5-33c1-4b6f-890c-7fe7600b36ee","final":false,"delta":" the"}
{"type":"te

In [74]:
result.__dict__

{'final_message': Message(id='80078235-cd4c-4fdc-af31-e3bf7148838b', role=<Role.ASSISTANT: 'assistant'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text="I'll search for gaming laptops on Amazon and provide you with 5 results."), ServerToolUseContent(content_block_type=<ContentBlockType.SERVER_TOOL_USE: 'server_tool_use'>, kwargs={'caller': {'type': 'direct'}}, tool_name='web_search', tool_id='srvtoolu_018sufNGFQcVzSxc6LsEu3KC', tool_input={'query': 'Amazon gaming laptops'}), ServerToolResultContent(content_block_type=<ContentBlockType.SERVER_TOOL_RESULT: 'server_tool_result'>, kwargs={'caller': {'type': 'direct'}}, tool_name='web_search_tool_result', tool_id='srvtoolu_018sufNGFQcVzSxc6LsEu3KC', tool_result=[{'encrypted_content': 'EqACCioIDRgCIiQxMDM4NWM5Ny0wZjFhLTQwMWEtYTBlOS00YjhmMzI2ODFhNTQSDKkjYIJC6xfOCV1/PRoMPjBVWrv6707oMh7gIjCjXmn/j1XjTtmrFx8XwSRwneysRCgD14539/kiHxm+Gu+kzOdnSkrkcaa11h+vWVwqowErrNB3nYOCPM6nc3ioZWUpIIkDVP7J5GexdguLBNtsXJvJkS

In [75]:
print(validate_answer(result.final_answer))

(True, '{"name": "ASUS ROG Strix G16 (2025) Gaming Laptop", "description": "16-inch FHD+ display with 165Hz/3ms refresh rate, powered by Intel Core i7 Processor 14650HX and NVIDIA GeForce RTX 5060 Laptop GPU. Features 16GB DDR5 RAM, 1TB Gen 4 SSD, Wi-Fi 7 connectivity, and Windows 11 Home. Includes advanced ROG Intelligent Cooling with vapor chamber technology, tri-fan design, and a full-surround RGB light bar with customizable Aura Sync lighting."}')


In [76]:
print(result.final_answer)

I'll search for gaming laptops on Amazon and provide you with 5 results.Let me fetch the Amazon gaming laptops page to get specific product details.Let me search for more specific gaming laptop products with model names.Let me search for a couple more gaming laptop brands to give you a good variety.Based on my search results, I've found information about various gaming laptops available on Amazon. Let me provide you with 5 diverse gaming laptop products in JSON format:

```json
[
  {
    "name": "ASUS ROG Strix G16 (2025) Gaming Laptop",
    "description": "16-inch FHD+ display with 165Hz/3ms refresh rate, powered by Intel Core i7 Processor 14650HX and NVIDIA GeForce RTX 5060 Laptop GPU. Features 16GB DDR5 RAM, 1TB Gen 4 SSD, Wi-Fi 7 connectivity, and Windows 11 Home. Includes advanced ROG Intelligent Cooling with vapor chamber technology, tri-fan design, and a full-surround RGB light bar with customizable Aura Sync lighting."
  },
  {
    "name": "Lenovo Legion Pro 7i Gen 10 Gaming La

# File Generation Agent + Rerun on same Agent Instance

In [ ]:
import asyncio
import os
from agent_base.providers.anthropic import AnthropicAgent, AnthropicLLMConfig
from agent_base.media_backend import LocalMediaBackend
from agent_base.storage import create_adapters

pg_config, pg_conv, pg_run = create_adapters(
    "postgres",
    connection_string=os.getenv("DATABASE_URL")
)

agent = AnthropicAgent(
    system_prompt="You are a helpful assistant that should help the user with their questions.",
    model="claude-sonnet-4-5",
    config=AnthropicLLMConfig(
        thinking_tokens=1024,
        max_tokens=64000,
        server_tools=[{
            "type": "code_execution_20250825",
            "name": "code_execution"
        }],
        beta_headers=["files-api-2025-04-14", "code-execution-2025-08-25"],
    ),
    media_backend=LocalMediaBackend(),
    config_adapter=pg_config,
    conversation_adapter=pg_conv,
    run_adapter=pg_run,
)

In [3]:
await agent.initialize()
agent_uuid = agent.agent_uuid
print(agent_uuid)

a118db51-5ff9-4580-bd96-14a21d0997d4


In [4]:
print(agent)

In [5]:
from agent_base.core.messages import Message
from agent_base.core.types import TextContent

prompt = Message.user([
    TextContent(text="Generate a sample invoice image for a hypothetical restuarant and return that image.")
])

In [6]:
result = await run_with_stream(agent, prompt, "json")

2026-03-05T16:48:03.495376Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":"The user wants"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":" me to generate a sample invoice image for"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":" a hypothetical restaurant. I"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":"'ll"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":" need to:"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":"\n\n1. Create the"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":" invoice"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":" content"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":" ("}
{"type":"thinking","agent":"a118db51-5ff

2026-03-05T16:48:45.900734Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkGw2mf8yjry75LURtNL/content?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-05T16:48:46.388219Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkGw2mf8yjry75LURtNL?beta=true "HTTP/1.1 200 OK" [httpx]


In [7]:
result.conversation_history

[Message(id='e520c772-5625-43c6-93dc-57770f6d77a1', role=<Role.USER: 'user'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='Generate a sample invoice image for a hypothetical restuarant and return that image.')], stop_reason=None, usage=None, provider='', model='', usage_kwargs={}),
 Message(id='a1b30832-6d61-45f6-a097-f10f1f36fecb', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking="The user wants me to generate a sample invoice image for a hypothetical restaurant. I'll need to:\n\n1. Create the invoice content (maybe as HTML or using a Python script with image generation)\n2. Convert it to an image\n3. Save it to the OUTPUT_DIR\n\nI think the best approach would be to use Python with PIL (Pillow) to create an invoice image. The container should have imagemagick and Python libraries available.\n\nLet me create a Python script that generates a restauran

In [8]:
result.generated_files

[MediaMetadata(media_id='120760cfe0cd4a2aad7d1dd2f7c443b7', media_mime_type='image/png', media_filename='restaurant_invoice.png', media_extension='png', media_size=67140, storage_type='local', storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/providers/anthropic/agent-media/a118db51-5ff9-4580-bd96-14a21d0997d4/120760cfe0cd4a2aad7d1dd2f7c443b7_restaurant_invoice.png', extras={'anthropic_file_id': 'file_011CYkGw2mf8yjry75LURtNL'})]

In [9]:
print(result.final_answer)

I'll create a sample restaurant invoice image for you using Python and PIL (Pillow). Let me generate a professional-looking invoice with typical restaurant details.Perfect! I've generated a professional restaurant invoice image for "The Golden Fork" restaurant. The invoice includes:

**Features:**
- Restaurant header with name, address, and contact information
- Invoice number and date details
- Server name and table number
- Itemized list of dishes and beverages with quantities and prices
- Subtotal, tax calculation (8%), and total amount
- Payment method information
- Professional layout with proper formatting

The invoice shows a typical dinner order with items like Caesar Salad, Grilled Salmon, Filet Mignon, wine, desserts, and more, totaling $196.40 including tax.

The image has been saved and is ready for you to download!


## Followup run on same agent instance

**Expected Behavior**
- Maintains context (self.messages)
- Resets conversation history.
- Should maintain file registry and stream both files at the end.

In [10]:
prompt_2 = Message.user([
    TextContent(text="Reduce some items from the invoice (modify the same file_id) and also create another version of the\
                invoice with the values in INR only.")
])

In [11]:
result_2 = await run_with_stream(agent, prompt_2, "json")

2026-03-05T16:48:48.205292Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":"The user wants me to:\n1"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":". Reduce"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":" some items from the invoice an"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":"d modify"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":" the same file ("}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":"over"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":"write the existing"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":" one"}
{"type":"thinking","agent":"a118db51-5ff9-4580-bd96-14a21d0997d4","final":false,"delta":")\n2. Create another"}
{"type":"thinking","agent":"a118db51-5f

2026-03-05T16:49:32.265657Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkGzHaUUaWa8JmXaLB5G/content?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-05T16:49:32.748809Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkGzHaUUaWa8JmXaLB5G?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-05T16:49:33.289184Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkGzEt1LzeY4QGUwwC8p/content?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-05T16:49:33.674689Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkGzEt1LzeY4QGUwwC8p?beta=true "HTTP/1.1 200 OK" [httpx]


In [12]:
result_2.conversation_history

[Message(id='d495b5e3-be34-4e51-a7ff-1444e0086cf2', role=<Role.USER: 'user'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='Reduce some items from the invoice (modify the same file_id) and also create another version of the                invoice with the values in INR only.')], stop_reason=None, usage=None, provider='', model='', usage_kwargs={}),
 Message(id='623cc151-a14a-4428-9d96-653f6eb1d742', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking="The user wants me to:\n1. Reduce some items from the invoice and modify the same file (overwrite the existing one)\n2. Create another version with values in INR (Indian Rupees) only\n\nI'll need to:\n1. Modify the Python script to have fewer items\n2. Generate the modified USD version and save it to OUTPUT_DIR\n3. Create another version with INR currency and save it to OUTPUT_DIR as well\n\nLet me create an 

In [13]:
result_2.generated_files

[MediaMetadata(media_id='61272eb6c6174fe9950dfece964d378d', media_mime_type='image/png', media_filename='restaurant_invoice.png', media_extension='png', media_size=61644, storage_type='local', storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/providers/anthropic/agent-media/a118db51-5ff9-4580-bd96-14a21d0997d4/61272eb6c6174fe9950dfece964d378d_restaurant_invoice.png', extras={'anthropic_file_id': 'file_011CYkGzHaUUaWa8JmXaLB5G'}),
 MediaMetadata(media_id='1db22691e11c48289288451ea9d148af', media_mime_type='image/png', media_filename='restaurant_invoice_INR.png', media_extension='png', media_size=63685, storage_type='local', storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/providers/anthropic/agent-media/a118db51-5ff9-4580-bd96-14a21d0997d4/1db22691e11c48289288451ea9d148af_restaurant_invoice_INR.png', extras={'anthropic_file_id': 'file_011CYkGzEt1LzeY4QGUwwC8p'})]

In [14]:
print(result_2.final_answer)

I'll modify the invoice to have fewer items and create both a USD version and an INR version for you.Perfect! I've successfully created both versions of the invoice with reduced items:

**Changes made:**
1. **Reduced items** from 8 to 4:
   - Grilled Salmon
   - Filet Mignon (Medium)
   - House Wine (Red)
   - Chocolate Lava Cake

2. **USD Version** (restaurant_invoice.png):
   - Original currency with $ symbol
   - Subtotal: $88.00
   - Tax: 8% ($7.04)
   - Total: $95.04

3. **INR Version** (restaurant_invoice_INR.png):
   - Converted to Indian Rupees (₹) using exchange rate of 83 INR per USD
   - Subtotal: ₹7,304.00
   - GST: 18% (₹1,314.72) - realistic restaurant tax rate in India
   - Total: ₹8,618.72
   - Shows "Currency: INR (Indian Rupees)" for clarity
   - Payment method shows "UPI/Card" which is common in India

Both invoices maintain the same professional design and have been saved with the same file ID as requested!


# Agent with Image/File + Thinking + Tool Calls (Server + Client) + Local File Backend + Storage Adapters

In [ ]:
import os
# Handle both running from project root and from notebook directory
test_image = "./currency_receipt_usd_jpy.png"
if not os.path.exists(test_image):
    test_image = "../../../currency_receipt_usd_jpy.png"

# Convert image to base64
import base64

with open(test_image, "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode('utf-8')

In [25]:
# Defining the tools
from agent_base.tools import tool

@tool
def get_rate(source: str, target: str) -> str:
    '''
    Get the exchange rate from source to target currency.
    Args:
        source: The source currency code. It can be INR, USD, or JPY.
        target: The target currency code. It can be INR, USD, or JPY.
    Returns:
        A string describing the exchange rate from source to target currency
    '''
    # For testing: return fixed mock rates
    mock_rates = {
        ("INR", "USD"): 0.012,
        ("USD", "INR"): 83.1,
        ("INR", "JPY"): 1.5,
        ("JPY", "INR"): 66.7,
        ("USD", "JPY"): 110.0,
        ("JPY", "USD"): 0.0091,
    }
    return f"The exchange rate from {source} to {target} is {mock_rates[(source, target)]}"

In [ ]:
import asyncio
import os

from agent_base.providers.anthropic import AnthropicAgent, AnthropicLLMConfig
from agent_base.media_backend import LocalMediaBackend
from agent_base.storage import create_adapters

media_backend = LocalMediaBackend()
config_adapter, conversation_adapter, run_adapter = create_adapters(
    "postgres",
    connection_string=os.getenv("DATABASE_URL")
)

agent = AnthropicAgent(
    system_prompt="You are a helpful assistant that should help the user with their questions.",
    model="claude-sonnet-4-5",
    config=AnthropicLLMConfig(
        thinking_tokens=1024,
        max_tokens=64000,
        server_tools=[{
            "type": "code_execution_20250825",
            "name": "code_execution"
        }],
        beta_headers=["files-api-2025-04-14", "code-execution-2025-08-25"],
    ),
    tools=[get_rate],
    media_backend=media_backend,
    config_adapter=config_adapter,
    conversation_adapter=conversation_adapter,
    run_adapter=run_adapter,
)

In [27]:
await agent.initialize()
agent_uuid = agent.agent_uuid
print(agent_uuid)

45f7db44-8302-451a-b029-49118dbbb884


In [28]:
print(agent)

In [29]:
from agent_base.core.messages import Message
from agent_base.core.types import ImageContent, TextContent

prompt_1 = Message.user([
    ImageContent(
        source_type="base64",
        media_type="image/png",
        data=base64_image,
    ),
    TextContent(text="I paid the receipt in USD. What is the amount in INR?"),
])

In [30]:
# import logging

# logging.basicConfig(level=logging.INFO)  # or configure handlers as you prefer
# # logging.getLogger("agent_base.providers.anthropic").setLevel(logging.DEBUG)

In [31]:
result = await run_with_stream(agent, prompt_1, "json")

2026-03-05T16:53:43.907684Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":"The user paid"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" $"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":"38"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":"."}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":"44"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" USD and wants"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" to know the equivalent"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" amount in INR ("}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":"Indian Rupees)."}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"del

2026-03-05T16:53:52.144125Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":"Based"}
{"type":"text","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" on the current"}
{"type":"text","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" exchange rate,"}
{"type":"text","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" your"}
{"type":"text","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" payment of **"}
{"type":"text","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":"$38.44 USD** equals"}
{"type":"text","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" approximately"}
{"type":"text","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" **"}
{"type":"text","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":"₹3,194"}
{"type":"text","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":".76"}
{"type":"text","agent

In [32]:
result.conversation_history

[Message(id='a48db966-6c75-4aff-9cb0-ec8cc250edc0', role=<Role.USER: 'user'>, content=[ImageContent(content_block_type=<ContentBlockType.IMAGE: 'image'>, kwargs={}, media_type='image/png', source_type='base64', data='...', filename=None), TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='I paid the receipt in USD. What is the amount in INR?')], stop_reason=None, usage=None, provider='', model='', usage_kwargs={}),
 Message(id='f522126d-dba8-4194-9a74-3e9f75717dd9', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking='The user paid $38.44 USD and wants to know the equivalent amount in INR (Indian Rupees).\n\nI need to use the get_rate function to get the exchange rate from USD to INR, and then calculate the amount.\n\nLet me call the get_rate function with source="USD" and target="INR".', signature='ErIDCkYICxgCKkB0tw5qzuHSUgkeLLAU77jkOJ1lSL5rCTeeg/Lo/vkt5J8H6oadJj58cq

In [33]:
result.generated_files

[]

In [34]:
print(result.final_answer)

Based on the current exchange rate, your payment of **$38.44 USD** equals approximately **₹3,194.76 INR** (38.44 × 83.1 = 3,194.764).


In [35]:
prompt_2 = Message.user([
    TextContent(text="Now generate a recepit image for the amount of various\
                items in INR only. Export and return the image to me.")
])

In [36]:
result_2 = await run_with_stream(agent, prompt_2, "json")

2026-03-05T16:53:54.241664Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":"The user wants me to create"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" a receipt image showing"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" the items"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" from"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" the receipt"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" but"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" with"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" prices"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-49118dbbb884","final":false,"delta":" in INR only. I need to"}
{"type":"thinking","agent":"45f7db44-8302-451a-b029-491

2026-03-05T16:54:28.139713Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkHNL88xq65dk3NuzH85/content?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-05T16:54:28.561072Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkHNL88xq65dk3NuzH85?beta=true "HTTP/1.1 200 OK" [httpx]


In [37]:
result_2.conversation_history

[Message(id='c65e2f81-edd0-4e86-a3ea-5de99cb903ac', role=<Role.USER: 'user'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='Now generate a recepit image for the amount of various                items in INR only. Export and return the image to me.')], stop_reason=None, usage=None, provider='', model='', usage_kwargs={}),
 Message(id='4f5347b4-c9e6-4da1-986c-154d82233dbe', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking="The user wants me to create a receipt image showing the items from the receipt but with prices in INR only. I need to:\n\n1. Convert all the USD prices to INR using the exchange rate I got (83.1)\n2. Create a receipt image similar to the original but showing INR prices\n\nLet me calculate the prices:\n- Coffee Beans 1kg: $18.50 × 83.1 = ₹1,537.35\n- Green Tea 200g: $9.75 × 83.1 = ₹810.23\n- Chocolate Bar: $2.99 × 83.1 = ₹248.47\n- Impo

In [38]:
result_2.generated_files

[MediaMetadata(media_id='70e186642a184011836a5dea0bdeb386', media_mime_type='image/png', media_filename='receipt_inr.png', media_extension='png', media_size=40089, storage_type='local', storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/providers/anthropic/agent-media/45f7db44-8302-451a-b029-49118dbbb884/70e186642a184011836a5dea0bdeb386_receipt_inr.png', extras={'anthropic_file_id': 'file_011CYkHNL88xq65dk3NuzH85'})]

In [39]:
print(result_2.final_answer)

Perfect! I've generated a receipt image with all the prices converted to INR. Here's what's on the receipt:

**Item Prices in INR:**
- Coffee Beans 1kg: ₹1,537.35
- Green Tea 200g: ₹810.23
- Chocolate Bar: ₹248.47
- Imported Snacks: ₹598.32

**TOTAL: ₹3,194.76**

The receipt image has been exported and is ready for download! 🧾


# Agent with Frontend Tools

In [3]:
from agent_base.tools import tool

# Define a frontend tool - this will be executed by the browser, not the server
@tool(executor="frontend")
def user_confirm(message: str) -> str:
    """Ask the user for yes/no confirmation before proceeding with an action.
    
    Use this tool when you need explicit user approval before taking an action
    that could have significant consequences.
    
    Args:
        message: The confirmation message to display to the user, explaining
                what action requires their approval.
    
    Returns:
        "yes" if user confirms, "no" if user declines
    """
    pass  # Never executed server-side - runs in browser


# Define a backend tool - executed on server
@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression.
    
    Args:
        expression: A mathematical expression to evaluate (e.g., "2 + 2")
    
    Returns:
        The result of the calculation as a string
    """
    try:
        # Safe eval for simple math
        allowed = set("0123456789+-*/().% ")
        if all(c in allowed for c in expression):
            result = eval(expression)
            return f"Result: {result}"
        return "Error: Invalid expression"
    except Exception as e:
        return f"Error: {str(e)}"
    
# Verify tool executor attributes
print(f"user_confirm executor: {user_confirm.__tool_executor__}")
print(f"calculate executor: {calculate.__tool_executor__}")

user_confirm executor: frontend
calculate executor: backend


In [4]:
import os
from agent_base.providers.anthropic import AnthropicAgent, AnthropicLLMConfig
from agent_base.storage import create_adapters

# Create postgres storage adapters for persistence across re-hydration
fs_config, fs_conv, fs_run = create_adapters(
    "postgres",
    connection_string=os.getenv("DATABASE_URL")
)

# Create agent with both backend and frontend tools
frontend_agent = AnthropicAgent(
    system_prompt="""You are a helpful assistant. When asked to perform calculations,
    use the calculate tool. When you get a result that seems significant (like any 
    number over 50), ask for user confirmation using the user_confirm tool before 
    reporting the final answer.""",
    model="claude-sonnet-4-5",
    config=AnthropicLLMConfig(
        thinking_tokens=1024,
        max_tokens=4096,
    ),
    tools=[calculate],  # Backend tools
    frontend_tools=[user_confirm],  # Frontend tools - executed in browser
    config_adapter=fs_config,
    conversation_adapter=fs_conv,
    run_adapter=fs_run,
)

print(frontend_agent)

## Running Agent - Pauses for Frontend Tool

When Claude decides to use a frontend tool (like `user_confirm`), the agent:
1. Executes any backend tools first
2. Persists state to DB
3. Returns early with `stop_reason="relay"`

In [5]:
prompt = "Calculate 25 * 4 for me."

In [6]:
result = await run_with_stream(frontend_agent, prompt)

print(f"\nStop reason: {result.stop_reason}")
print(f"Total steps: {result.total_steps}")

{"type":"meta_init","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":true,"delta":"{\"format\":\"json\",\"user_query\":\"Calculate 25 * 4 for me.\",\"agent_uuid\":\"0bf66af2-8052-45a9-beab-2775f2d1d357\",\"parent_agent_uuid\":null,\"model\":\"claude-sonnet-4-5\"}"}


2026-03-29T07:21:44.976932Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"thinking","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":false,"delta":"The user wants me to calculate 25"}
{"type":"thinking","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":false,"delta":" * 4. I should use the calculate tool for this.\n\nLet me calculate it first, then check"}
{"type":"thinking","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":false,"delta":" if the result is over 50 to determine if I need user confirmation.\n\n25 * 4 = 100,"}
{"type":"thinking","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":false,"delta":" which is over 50, so I'll need to ask for confirmation before reporting the final answer."}
{"type":"thinking","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":true,"delta":""}
{"type":"tool_call","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":true,"delta":"{\"expression\": \"25 * 4\"}","id":"toolu_01XCymh8fMdzgycSidh3Pisk","name":"calculate"}


2026-03-29T07:21:48.586876Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"tool_call","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":true,"delta":"{\"message\": \"The calculation result is 100, which is a significant number (over 50). Should I proceed with reporting this as the final answer?\"}","id":"toolu_01TSm8oYNYAjgDArFLTZDPt7","name":"user_confirm"}


2026-03-29T07:21:49.687807Z [info     ] Created PostgreSQL connection pool [agent_base.storage.adapters.postgres] max_size=10


{"type":"awaiting_frontend_tools","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":true,"delta":"{\"tools\":[{\"tool_use_id\":\"toolu_01TSm8oYNYAjgDArFLTZDPt7\",\"name\":\"user_confirm\",\"input\":{\"message\":\"The calculation result is 100, which is a significant number (over 50). Should I proceed with reporting this as the final answer?\"}}]}"}

Stop reason: relay
Total steps: 2


In [7]:
# Inspect the paused state
pending_relay = frontend_agent.agent_config.pending_relay
print(f"Has pending relay: {pending_relay is not None}")
if pending_relay:
    print(f"Pending frontend calls: {len(pending_relay.frontend_calls)}")
    for call in pending_relay.frontend_calls:
        print(f"  - name: {call.name}, tool_id: {call.tool_id}, input: {call.input}")
    print(f"Completed backend results: {len(pending_relay.completed_results)}")
print(f"Current step: {frontend_agent.agent_config.current_step}")

Has pending relay: True
Pending frontend calls: 1
  - name: user_confirm, tool_id: toolu_01TSm8oYNYAjgDArFLTZDPt7, input: {'message': 'The calculation result is 100, which is a significant number (over 50). Should I proceed with reporting this as the final answer?'}
Completed backend results: 0
Current step: 2


## Re-hydrating Agent and Submitting Frontend Tool Results

In a real scenario (like with FastAPI), the agent instance would be lost after the first request.
The state is persisted to DB, so we can re-hydrate using the agent_uuid and continue.

In [8]:
# Save the UUID before "losing" the agent instance
saved_uuid = frontend_agent.agent_uuid
print(f"Saved UUID: {saved_uuid}")

# Get the pending tool info (in a real scenario this comes from the frontend)
pending_call = pending_relay.frontend_calls[0]
tool_use_id = pending_call.tool_id
print(f"Tool use ID: {tool_use_id}")
print(f"Tool name: {pending_call.name}")
print(f"Tool input: {pending_call.input}")

Saved UUID: 0bf66af2-8052-45a9-beab-2775f2d1d357
Tool use ID: toolu_01TSm8oYNYAjgDArFLTZDPt7
Tool name: user_confirm
Tool input: {'message': 'The calculation result is 100, which is a significant number (over 50). Should I proceed with reporting this as the final answer?'}


In [9]:
# Simulate re-hydration: Create new agent instance with same UUID
# (In FastAPI, this happens in the /agent/tool_results endpoint)
rehydrated_agent = AnthropicAgent(
    system_prompt=frontend_agent.system_prompt,
    model=frontend_agent.model,
    config=frontend_agent.config,
    tools=[calculate],
    frontend_tools=[user_confirm],
    config_adapter=fs_config,
    conversation_adapter=fs_conv,
    run_adapter=fs_run,
    agent_uuid=saved_uuid,  # This triggers loading state from storage
)

print(f"Re-hydrated agent UUID: {rehydrated_agent.agent_uuid}")

Re-hydrated agent UUID: 0bf66af2-8052-45a9-beab-2775f2d1d357


In [10]:
from agent_base.core.types import ToolResultContent

# Submit frontend tool results and continue the agent
# (Simulating user clicking "yes" in the browser)
relay_results = [
    ToolResultContent(
        tool_id=tool_use_id,
        tool_result="yes",  # User confirmed
        is_error=False,
    )
]

# Continue the agent with frontend tool results
queue2: asyncio.Queue[str | None] = asyncio.Queue()

async def _drain():
    while True:
        chunk = await queue2.get()
        if chunk is None:
            break
        print(chunk, end="\n", flush=True)
        queue2.task_done()

printer2 = asyncio.create_task(_drain())
result2 = await rehydrated_agent.resume_with_relay_results(relay_results, queue2)
await queue2.put(None)
await printer2

print(f"\n\nStop reason: {result2.stop_reason}")
print(f"Total steps: {result2.total_steps}")

{"type":"meta_init","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":true,"delta":"{\"format\":\"json\",\"user_query\":\"{\\\"id\\\": \\\"576ddfab-3d19-42f8-9999-e93c8d7ef41b\\\", \\\"role\\\": \\\"user\\\", \\\"content\\\": [{\\\"content_block_type\\\": \\\"tool_result\\\", \\\"tool_name\\\": \\\"\\\", \\\"tool_id\\\": \\\"toolu_01TSm8oYNYAjgDArFLTZDPt7\\\", \\\"tool_result\\\": \\\"yes\\\", \\\"is_error\\\": false, \\\"kwargs\\\": {}}], \\\"stop_reason\\\": null, \\\"usage\\\": null, \\\"provider\\\": \\\"\\\", \\\"model\\\": \\\"\\\", \\\"usage_kwargs\\\": {}}\",\"agent_uuid\":\"0bf66af2-8052-45a9-beab-2775f2d1d357\",\"parent_agent_uuid\":null,\"model\":\"claude-sonnet-4-5\"}"}


2026-03-29T07:22:04.011847Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":false,"delta":"The"}
{"type":"text","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":false,"delta":" result of 25 * 4 is **100**."}
{"type":"text","agent":"0bf66af2-8052-45a9-beab-2775f2d1d357","final":true,"delta":""}


Stop reason: end_turn
Total steps: 3


In [11]:
# Show the final answer
print("Final Answer:")
print(result2.final_answer)

Final Answer:
The result of 25 * 4 is **100**.


# Subagent Test

## Travel Planning with Specialist Subagents
Orchestrator delegates flight searches to `flight_finder` and hotel searches to `hotel_finder`, then synthesizes results.

In [13]:
# Define specialist tools for subagents
from agent_base.tools import tool

@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search for available flights between two cities on a given date.

    Args:
        origin: The departure city, e.g. "New York"
        destination: The arrival city, e.g. "London"
        date: The travel date in YYYY-MM-DD format

    Returns:
        A string listing available flights with prices
    """
    flights = {
        ("New York", "Tokyo"): [
            {"airline": "ANA", "departure": "10:00", "arrival": "14:00+1", "price": "$1,200"},
            {"airline": "JAL", "departure": "13:30", "arrival": "17:30+1", "price": "$1,350"},
        ],
        ("Tokyo", "New York"): [
            {"airline": "ANA", "departure": "17:00", "arrival": "16:00", "price": "$1,180"},
            {"airline": "United", "departure": "19:30", "arrival": "18:30", "price": "$1,050"},
        ],
    }
    key = (origin, destination)
    if key not in flights:
        return f"No flights found from {origin} to {destination} on {date}."
    results = [f"Flights from {origin} to {destination} on {date}:"]
    for f in flights[key]:
        results.append(f"  - {f['airline']}: departs {f['departure']}, arrives {f['arrival']}, {f['price']}")
    return "\n".join(results)


@tool
def search_hotels(city: str, checkin: str, checkout: str) -> str:
    """Search for available hotels in a city for given dates.

    Args:
        city: The city to search hotels in, e.g. "Tokyo"
        checkin: Check-in date in YYYY-MM-DD format
        checkout: Check-out date in YYYY-MM-DD format

    Returns:
        A string listing available hotels with nightly rates
    """
    hotels = {
        "Tokyo": [
            {"name": "Park Hyatt Tokyo", "rating": "5-star", "price": "$450/night"},
            {"name": "Shinjuku Granbell", "rating": "3-star", "price": "$120/night"},
            {"name": "Hotel Gracery Shinjuku", "rating": "4-star", "price": "$200/night"},
        ],
    }
    if city not in hotels:
        return f"No hotels found in {city} for {checkin} to {checkout}."
    results = [f"Hotels in {city} ({checkin} to {checkout}):"]
    for h in hotels[city]:
        results.append(f"  - {h['name']} ({h['rating']}): {h['price']}")
    return "\n".join(results)

In [14]:
import asyncio
import os
from agent_base.providers.anthropic import AnthropicAgent
from agent_base.storage import create_adapters

# Postgres storage adapters
sa_config, sa_conv, sa_run = create_adapters(
    "postgres",
    connection_string=os.getenv("DATABASE_URL")
)
await sa_config.connect()
await sa_conv.connect()
await sa_run.connect()

# Specialist 1: Flight Finder
flight_agent = AnthropicAgent(
    system_prompt=(
        "You are a flight search specialist. Use the search_flights tool to find "
        "flights for the user. Always search for the exact origin, destination, and "
        "date provided. Present the results clearly."
    ),
    description="Searches for available flights between cities on a specific date",
    model="claude-sonnet-4-5",
    tools=[search_flights],
    config_adapter=sa_config,
    conversation_adapter=sa_conv,
    run_adapter=sa_run,
)

# Specialist 2: Hotel Finder
hotel_agent = AnthropicAgent(
    system_prompt=(
        "You are a hotel search specialist. Use the search_hotels tool to find "
        "hotels for the user. Always search for the exact city and dates provided. "
        "Present the results clearly."
    ),
    description="Searches for available hotels in a city for specific check-in/check-out dates",
    model="claude-sonnet-4-5",
    tools=[search_hotels],
    config_adapter=sa_config,
    conversation_adapter=sa_conv,
    run_adapter=sa_run,
)

# Orchestrator: delegates to specialists via SubAgentTool
orchestrator = AnthropicAgent(
    system_prompt=(
        "You are a travel planning coordinator. When the user asks to plan a trip, "
        "you MUST delegate flight searches to the flight_finder subagent and hotel "
        "searches to the hotel_finder subagent. After receiving results from both, "
        "synthesize them into a concise travel plan with total estimated costs."
    ),
    model="claude-sonnet-4-5",
    subagents={
        "flight_finder": flight_agent,
        "hotel_finder": hotel_agent,
    },
    config_adapter=sa_config,
    conversation_adapter=sa_conv,
    run_adapter=sa_run,
)

print(f"Orchestrator UUID: {orchestrator.agent_uuid}")

2026-03-29T06:22:29.419506Z [info     ] Created PostgreSQL connection pool [agent_base.storage.adapters.postgres] max_size=10


Orchestrator UUID: None


In [15]:
prompt = (
    "Plan a 5-night trip from New York to Tokyo. "
    "I need flights departing 2025-03-15 and returning 2025-03-20, "
    "plus a hotel for those dates. Find the best options and give me a total cost estimate."
)

result = await run_with_stream(orchestrator, prompt)

{"type":"meta_init","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":true,"delta":"{\"format\":\"json\",\"user_query\":\"Plan a 5-night trip from New York to Tokyo. I need flights departing 2025-03-15 and returning 2025-03-20, plus a hotel for those dates. Find the best options and give me a total cost estimate.\",\"agent_uuid\":\"d28727f5-19fd-44bc-80a9-defccb6e1989\",\"parent_agent_uuid\":null,\"model\":\"claude-sonnet-4-5\"}"}


2026-03-29T06:22:33.957538Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":false,"delta":"I'll help you plan your"}
{"type":"text","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":false,"delta":" 5-night trip from New York to Tokyo! Let me search for flights and hotels for your dates."}
{"type":"text","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":true,"delta":""}
{"type":"tool_call","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":true,"delta":"{\"agent_name\": \"flight_finder\", \"task\": \"Find available flights from New York to Tokyo departing on 2025-03-15, and return flights from Tokyo to New York on 2025-03-20. Provide the best options with prices.\"}","id":"toolu_01DBfxgscCqZ7cAK7eQ9jcB8","name":"spawn_subagent"}
{"type":"tool_call","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":true,"delta":"{\"agent_name\": \"hotel_finder\", \"task\": \"Find available hotels in Tokyo for check-in on 2025-03-15 and check-out on 2025-03-20 (5 nights). Provide the best opt

2026-03-29T06:22:37.337797Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-29T06:22:37.341523Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"5b6548e3-e9ea-495f-b8b5-97a181a91d47","final":false,"delta":"I'll search for flights"}
{"type":"text","agent":"5b6548e3-e9ea-495f-b8b5-97a181a91d47","final":false,"delta":" in both directions for your trip between New York and Tokyo."}
{"type":"text","agent":"5b6548e3-e9ea-495f-b8b5-97a181a91d47","final":true,"delta":""}
{"type":"tool_call","agent":"28966086-6761-4a33-8fcc-2d429a07d26b","final":true,"delta":"{\"city\": \"Tokyo\", \"checkin\": \"2025-03-15\", \"checkout\": \"2025-03-20\"}","id":"toolu_01BUHVUi9n2Qg4WsnHR8AvTv","name":"search_hotels"}
{"type":"tool_call","agent":"5b6548e3-e9ea-495f-b8b5-97a181a91d47","final":true,"delta":"{\"origin\": \"New York\", \"destination\": \"Tokyo\", \"date\": \"2025-03-15\"}","id":"toolu_01AReoXm3rWUdp533NFDwNZs","name":"search_flights"}
{"type":"tool_call","agent":"5b6548e3-e9ea-495f-b8b5-97a181a91d47","final":true,"delta":"{\"origin\": \"Tokyo\", \"destination\": \"New York\", \"date\": \"2025-03-20\"}","id":"toolu_01V

...

{"type":"text","agent":"28966086-6761-4a33-8fcc-2d429a07d26b","final":false,"delta":"I found"}
{"type":"text","agent":"28966086-6761-4a33-8fcc-2d429a07d26b","final":false,"delta":" 3 available hotels in Tokyo for your 5-night stay (March 15-20, 2025). Here are your options:\n\n## Best"}


2026-03-29T06:22:40.247267Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"5b6548e3-e9ea-495f-b8b5-97a181a91d47","final":false,"delta":"Perfect"}
{"type":"text","agent":"28966086-6761-4a33-8fcc-2d429a07d26b","final":false,"delta":" Hotel Options in Tokyo\n\n### 🌟 **Park Hyatt Tokyo** (5-star)\n- **Price:** $450/night\n-"}
{"type":"text","agent":"5b6548e3-e9ea-495f-b8b5-97a181a91d47","final":false,"delta":"! I found several flight options for your round trip between New York and Tokyo. Here are the results:\n\n## **Outbound Flights:"}
{"type":"text","agent":"28966086-6761-4a33-8fcc-2d429a07d26b","final":false,"delta":" **Total for 5 nights:** $2,250\n- Luxury accommodation with top-tier amenities\n\n### 🏨 **Hotel"}
{"type":"text","agent":"5b6548e3-e9ea-495f-b8b5-97a181a91d47","final":false,"delta":" New York → Tokyo (March 15, 2025)**\n\n1. **ANA (All Nippon Airways)** - **$"}
{"type":"text","agent":"28966086-6761-4a33-8fcc-2d429a07d26b","final":false,"delta":" Gracery Shinjuku** (4-star)\n- **Price:** $200/night\n- **Total for 5 nights

2026-03-29T06:22:46.349607Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":false,"delta":"Perfect"}
{"type":"text","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":false,"delta":"! I've found great options for your 5-night trip to Tokyo. Here's"}
{"type":"text","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":false,"delta":" your complete travel plan:\n\n## 🗼 **New York to Tokyo Trip Summary"}
{"type":"text","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":false,"delta":"**\n**Dates:** March 15-20, 2025 (5 nights)\n\n---\n\n### "}
{"type":"text","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":false,"delta":"✈️ **Recommended Flights**\n\n**Outbound:** ANA (All Nippon Airways) - March 15\n- Departs"}
{"type":"text","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":false,"delta":": 10:00 AM → Arrives: 2:00 PM (next day)\n- **Cost: $1,200**\n\n**Return:** United Airlines - March 20"}
{"type":"text","agent":"d28727f5-19fd-44bc-80a9-defccb6e1989","final":false,"delta

In [16]:
# Show the final synthesized travel plan
print("=== Final Answer ===")
print(result.final_answer)

=== Final Answer ===
Perfect! I've found great options for your 5-night trip to Tokyo. Here's your complete travel plan:

## 🗼 **New York to Tokyo Trip Summary**
**Dates:** March 15-20, 2025 (5 nights)

---

### ✈️ **Recommended Flights**

**Outbound:** ANA (All Nippon Airways) - March 15
- Departs: 10:00 AM → Arrives: 2:00 PM (next day)
- **Cost: $1,200**

**Return:** United Airlines - March 20
- Departs: 7:30 PM → Arrives: 6:30 PM (same day)
- **Cost: $1,050**

**Flight Total: $2,250**

---

### 🏨 **Hotel Options**

**Budget Option:** Shinjuku Granbell (3-star)
- $120/night × 5 nights = **$600**
- **Trip Total: $2,850**

**Mid-Range Option:** Hotel Gracery Shinjuku (4-star) ⭐ *Recommended*
- $200/night × 5 nights = **$1,000**
- **Trip Total: $3,250**

**Luxury Option:** Park Hyatt Tokyo (5-star)
- $450/night × 5 nights = **$2,250**
- **Trip Total: $4,500**

---

## 💰 **Total Cost Estimates**
- **Budget Trip:** $2,850 (flights + Shinjuku Granbell)
- **Mid-Range Trip:** $3,250 (flights

In [17]:
# Inspect execution details
print(f"Stop reason: {result.stop_reason}")
print(f"Total steps: {result.total_steps}")
print(f"Model: {result.model}")
if result.cost:
    print(f"Cost: {result.cost}")
if result.cumulative_usage:
    print(f"Cumulative usage: {result.cumulative_usage}")

Stop reason: end_turn
Total steps: 2
Model: claude-sonnet-4-5
Cost: CostBreakdown(total_cost=0.020721, breakdown={'input_cost': 0.002616, 'output_cost': 0.010815, 'cache_write_5m_cost': 0.00729, 'cache_write_1h_cost': 0.0, 'cache_read_cost': 0.0, 'web_search_cost': 0.0, 'web_fetch_cost': 0.0})
Cumulative usage: Usage(input_tokens=879, output_tokens=721, cache_write_tokens=1944, cache_read_tokens=None, thinking_tokens=None, raw_usage={})


# Agent with Anthropic Skills

Test the Anthropic Agent Skills integration (xlsx, pptx, docx, pdf).
Skills run in Anthropic's server-side code execution container and can generate documents.

In [18]:
import asyncio
import os
from agent_base.providers.anthropic import AnthropicAgent, AnthropicLLMConfig
from agent_base.storage import create_adapters

# Postgres adapters for persistence
fs_config, fs_conv, fs_run = create_adapters(
    "postgres",
    connection_string=os.getenv("DATABASE_URL")
)

await fs_config.connect()
await fs_conv.connect()
await fs_run.connect()

agent = AnthropicAgent(
    system_prompt="You are a helpful assistant that can generate documents using Anthropic Skills.",
    model="claude-sonnet-4-5",
    config=AnthropicLLMConfig(
        max_tokens=64000,
        server_tools=[
            {"type": "code_execution_20250825", "name": "code_execution"},
        ],
        skills=[
            {"type": "anthropic", "skill_id": "xlsx", "version": "latest"},
            {"type": "anthropic", "skill_id": "pptx", "version": "latest"},
        ],
        beta_headers=[
            "code-execution-2025-08-25",
            "skills-2025-10-02",
            "files-api-2025-04-14",
        ],
    ),
    config_adapter=fs_config,
    conversation_adapter=fs_conv,
    run_adapter=fs_run,
    stream_meta_history_and_tool_results=False,
)

2026-03-29T06:23:11.917377Z [info     ] Created PostgreSQL connection pool [agent_base.storage.adapters.postgres] max_size=10


In [19]:
await agent.initialize()
agent_uuid = agent.agent_uuid
print(agent_uuid)

28d43ef6-24cf-4c72-9eee-c33ca79663d0


In [20]:
print(agent)

## Skills: Generate Excel Spreadsheet

In [22]:
prompt = "Create an Excel spreadsheet with a simple monthly budget: categories (Rent, Food, Transport, Entertainment, Savings) and amounts for 3 months."

import nest_asyncio
nest_asyncio.apply()

result = await run_with_stream(agent, prompt)

2026-03-29T06:23:56.462311Z [warning  ] defensive_sanitize_repaired_chain [agent_base.providers.anthropic.provider] original_count=2 sanitized_count=1


{"type":"meta_init","agent":"28d43ef6-24cf-4c72-9eee-c33ca79663d0","final":true,"delta":"{\"format\":\"json\",\"user_query\":\"Create an Excel spreadsheet with a simple monthly budget: categories (Rent, Food, Transport, Entertainment, Savings) and amounts for 3 months.\",\"agent_uuid\":\"28d43ef6-24cf-4c72-9eee-c33ca79663d0\",\"parent_agent_uuid\":null,\"model\":\"claude-sonnet-4-5\"}"}


2026-03-29T06:23:57.509365Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"28d43ef6-24cf-4c72-9eee-c33ca79663d0","final":false,"delta":"I'll help"}
{"type":"text","agent":"28d43ef6-24cf-4c72-9eee-c33ca79663d0","final":false,"delta":" you create an Excel spreadsheet with a monthly budget. Let me first read the Excel skill instructions."}
{"type":"text","agent":"28d43ef6-24cf-4c72-9eee-c33ca79663d0","final":true,"delta":""}
{"type":"server_tool_call","agent":"28d43ef6-24cf-4c72-9eee-c33ca79663d0","final":true,"delta":"{\"command\": \"view\", \"path\": \"/skills/xlsx/SKILL.md\"}","id":"srvtoolu_01HWksejQKkVzr9cVcENtGFN","name":"text_editor_code_execution"}
{"type":"text","agent":"28d43ef6-24cf-4c72-9eee-c33ca79663d0","final":false,"delta":"Now"}
{"type":"text","agent":"28d43ef6-24cf-4c72-9eee-c33ca79663d0","final":false,"delta":" I'll create a monthly budget spreadsheet with the categories"}
{"type":"text","agent":"28d43ef6-24cf-4c72-9eee-c33ca79663d0","final":false,"delta":" and amounts for 3 months using openpyxl:"}
{"type":"text","agen

2026-03-29T06:24:34.608753Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CZWtk9QKuAnnpgQPC2Y8p/content?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-29T06:24:35.089104Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CZWtk9QKuAnnpgQPC2Y8p?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"meta_files","agent":"28d43ef6-24cf-4c72-9eee-c33ca79663d0","final":true,"delta":"{\"files\":[{\"media_id\":\"242bea5be8944c37b19385440b9967f1\",\"media_mime_type\":\"application/vnd.openxmlformats-officedocument.spreadsheetml.sheet\",\"media_filename\":\"monthly_budget.xlsx\",\"media_extension\":\"xlsx\",\"media_size\":5852,\"storage_type\":\"local\",\"storage_location\":\"/Users/aurosoni/conductor/workspaces/anthropic_agent/bandung/agent_base/providers/anthropic/agent-media/28d43ef6-24cf-4c72-9eee-c33ca79663d0/242bea5be8944c37b19385440b9967f1_monthly_budget.xlsx\",\"url\":\"file:///Users/aurosoni/conductor/workspaces/anthropic_agent/bandung/agent_base/providers/anthropic/agent-media/28d43ef6-24cf-4c72-9eee-c33ca79663d0/242bea5be8944c37b19385440b9967f1_monthly_budget.xlsx\",\"extras\":{\"anthropic_file_id\":\"file_011CZWtk9QKuAnnpgQPC2Y8p\"}}]}"}


In [23]:
print(f"Stop reason: {result.stop_reason}")
print(f"Total steps: {result.total_steps}")
print(f"Generated File: {result.generated_files}")
print(f"Final answer:\n{result.final_answer}")

Stop reason: end_turn
Total steps: 1
Generated File: [MediaMetadata(media_id='242bea5be8944c37b19385440b9967f1', media_mime_type='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet', media_filename='monthly_budget.xlsx', media_extension='xlsx', media_size=5852, storage_type='local', storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/bandung/agent_base/providers/anthropic/agent-media/28d43ef6-24cf-4c72-9eee-c33ca79663d0/242bea5be8944c37b19385440b9967f1_monthly_budget.xlsx', url='file:///Users/aurosoni/conductor/workspaces/anthropic_agent/bandung/agent_base/providers/anthropic/agent-media/28d43ef6-24cf-4c72-9eee-c33ca79663d0/242bea5be8944c37b19385440b9967f1_monthly_budget.xlsx', extras={'anthropic_file_id': 'file_011CZWtk9QKuAnnpgQPC2Y8p'})]
Final answer:
I'll help you create an Excel spreadsheet with a monthly budget. Let me first read the Excel skill instructions.Now I'll create a monthly budget spreadsheet with the categories and amounts for 3 months

## Skills: Generate PowerPoint Presentation

In [ ]:
prompt_pptx = "Create a 3-slide PowerPoint presentation about renewable energy: Title slide, Key Benefits slide, and Future Outlook slide."

result_pptx = await run_with_stream(agent, prompt_pptx)

2026-03-05T16:58:01.185886Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"0be91de0-a3e8-4231-8355-69755463ace1","final":false,"delta":"I'll help"}
{"type":"text","agent":"0be91de0-a3e8-4231-8355-69755463ace1","final":false,"delta":" you create a Power"}
{"type":"text","agent":"0be91de0-a3e8-4231-8355-69755463ace1","final":false,"delta":"Point presentation about renewable energy. Let me"}
{"type":"text","agent":"0be91de0-a3e8-4231-8355-69755463ace1","final":false,"delta":" first"}
{"type":"text","agent":"0be91de0-a3e8-4231-8355-69755463ace1","final":false,"delta":" read"}
{"type":"text","agent":"0be91de0-a3e8-4231-8355-69755463ace1","final":false,"delta":" the p"}
{"type":"text","agent":"0be91de0-a3e8-4231-8355-69755463ace1","final":false,"delta":"ptx skill"}
{"type":"text","agent":"0be91de0-a3e8-4231-8355-69755463ace1","final":false,"delta":" to ensure"}
{"type":"text","agent":"0be91de0-a3e8-4231-8355-69755463ace1","final":false,"delta":" I follow"}
{"type":"text","agent":"0be91de0-a3e8-4231-8355-69755463ace1","final":false,"delta":" 

2026-03-05T17:00:23.416666Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkHp4orrP9nMSMjGhqhu/content?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-05T17:00:24.072663Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkHp4orrP9nMSMjGhqhu?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-05T17:00:24.646717Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkHp7RsWSnrAzdefd76Q/content?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-05T17:00:25.364611Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkHp7RsWSnrAzdefd76Q?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-05T17:00:26.097557Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkHoybbDVE4914VDtXxb/content?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-05T17:00:26.937134Z [info     ] HTTP Request: GET https://api.anthropic.com/v1/files/file_011CYkHoybbDVE4914VDtXxb?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-05T17:00:27.53

In [ ]:
print(f"Stop reason: {result_pptx.stop_reason}")
print(f"Total steps: {result_pptx.total_steps}")
print(f"Generated File: {result_pptx.generated_files}")
print(f"Final answer:\n{result_pptx.final_answer}")

Stop reason: end_turn
Total steps: 1
Generated File: [MediaMetadata(media_id='d74c0522f8fc4f76844fc127eae7c6b7', media_mime_type='image/jpeg', media_filename='slide-2.jpg', media_extension='jpg', media_size=75025, storage_type='local', storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/providers/anthropic/agent-media/0be91de0-a3e8-4231-8355-69755463ace1/d74c0522f8fc4f76844fc127eae7c6b7_slide-2.jpg', extras={'anthropic_file_id': 'file_011CYkHp4orrP9nMSMjGhqhu'}), MediaMetadata(media_id='270b8c741ae143d9af2e0030fa764b42', media_mime_type='image/jpeg', media_filename='slide-1.jpg', media_extension='jpg', media_size=42674, storage_type='local', storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/providers/anthropic/agent-media/0be91de0-a3e8-4231-8355-69755463ace1/270b8c741ae143d9af2e0030fa764b42_slide-1.jpg', extras={'anthropic_file_id': 'file_011CYkHp7RsWSnrAzdefd76Q'}), MediaMetadata(media_id='8e1045f5e58a4c6092e3

# Context Externalization Tests

Test file-backed context externalization with strict 10,000-token budgets. These scenarios still keep regular long-chat compaction enabled, but the assertions here focus on `.context/` references for oversized newest-message payloads.

In [3]:

from functools import lru_cache
from urllib.request import urlopen
from pprint import pprint
from agent_base.core.types import TextContent, ToolResultContent
from agent_base.providers.anthropic import AnthropicAgent, ExternalizationConfig
from agent_base.providers.anthropic.compaction import CompactionConfig
from agent_base.storage import create_adapters
from agent_base.tools import tool


# Keep the externalization section runnable after a fresh kernel restart.
config_adapter, conversation_adapter, run_adapter = create_adapters("memory")

LARGE_DOCUMENT_TITLE = "War and Peace"
LARGE_DOCUMENT_URL = "https://www.gutenberg.org/cache/epub/2600/pg2600.txt"
FALLBACK_DOCUMENT_TEXT = """
BOOK ONE: 1805

"Well, Prince, so Genoa and Lucca are now just family estates of the Buonapartes.
But I warn you, if you don't tell me that this means war, if you still try to defend
the infamies and horrors perpetrated by that Antichrist-I really believe he is Antichrist-
I will have nothing more to do with you and you are no longer my friend, no longer my
'faithful slave,' as you call yourself!"

Anna Pavlovna's words came with composure and confidence. She spoke in French, as she
always did when feeling especially animated, and her familiar visitor answered in the
same language and tone as though the whole of Europe could be neatly arranged between
one drawing-room call and the next.
""".strip()


@lru_cache(maxsize=1)
def load_large_document() -> str:
    """Fetch and cache a large public-domain document for compaction demos."""
    try:
        with urlopen(LARGE_DOCUMENT_URL, timeout=20) as response:
            raw_text = response.read().decode("utf-8")
    except Exception:
        raw_text = FALLBACK_DOCUMENT_TEXT

    start_marker = f"*** START OF THE PROJECT GUTENBERG EBOOK {LARGE_DOCUMENT_TITLE.upper()} ***"
    end_marker = f"*** END OF THE PROJECT GUTENBERG EBOOK {LARGE_DOCUMENT_TITLE.upper()} ***"
    if start_marker in raw_text:
        raw_text = raw_text.split(start_marker, 1)[1].lstrip()
    if end_marker in raw_text:
        raw_text = raw_text.split(end_marker, 1)[0].rstrip()
    return raw_text


def make_filler(tokens: int = 12_000) -> str:
    """Return an approximately sized excerpt from a cached real document."""
    header = f"Document: {LARGE_DOCUMENT_TITLE}\nSource: {LARGE_DOCUMENT_URL}\n\n"
    target_chars = max(tokens * 4 - len(header), 0)
    document = load_large_document()
    if len(document) < target_chars:
        repeats = (target_chars // max(len(document), 1)) + 1
        document = ((document + "\n\n") * repeats).strip()
    return header + document[:target_chars]


@tool
def large_result_tool(query: str) -> str:
    """Look up information about a topic. Returns detailed results.

    Args:
        query: The search query

    Returns:
        Detailed information about the query
    """
    return f"Result for '{query}': " + make_filler(12_000)


@tool
def medium_result_tool(topic: str) -> str:
    """Fetch a medium-sized report on a topic.

    Args:
        topic: The topic to report on

    Returns:
        A medium-length report
    """
    # ~4k tokens each — individually under the per-result threshold,
    # but 3 parallel calls = ~12k tokens which exceeds the combined budget.
    return f"Report on '{topic}': " + make_filler(4_000)


COMPACTION_TOOLS = [large_result_tool, medium_result_tool]

STRICT_COMPACTION = CompactionConfig(
    threshold_tokens=5000,
    preserve_recent_tokens=2_000,
)

STRICT_EXTERNALIZATION = ExternalizationConfig(
    max_prompt_tokens=10_000,
    max_tool_result_tokens=10_000,
    max_combined_tool_result_tokens=10_000,
)


def extract_text_payload(blocks: list[TextContent]) -> str:
    return "".join(block.text for block in blocks if isinstance(block, TextContent))


def tool_result_text(block: ToolResultContent) -> str:
    if isinstance(block.tool_result, str):
        return block.tool_result
    return extract_text_payload(block.tool_result)


def first_tool_result_message(messages):
    return next(
        msg for msg in messages
        if any(isinstance(block, ToolResultContent) for block in msg.content)
    )

## Scenario 1: User Prompt too large

In [4]:
from agent_base.common_tools import GlobFileSearchTool, ListDirTreeTool, ReadFileTool

# Scenario 1: Initial user prompt is too large and should be externalized
agent_compact = AnthropicAgent(
    system_prompt="You are a helpful assistant.",
    model="claude-sonnet-4-5",
    compaction_config=STRICT_COMPACTION,
    externalization_config=STRICT_EXTERNALIZATION,
    tools=[
        ReadFileTool().get_tool(),
        ListDirTreeTool().get_tool(),
        GlobFileSearchTool().get_tool(),
    ]+ COMPACTION_TOOLS,
    config_adapter=config_adapter,
    conversation_adapter=conversation_adapter,
    run_adapter=run_adapter,
)
await agent_compact.initialize()

big_prompt = (
    "Please summarize the following text:\n\n"
    + make_filler(12_000)
)

result_1 = await run_with_stream(agent_compact, big_prompt)
print(f"\n--- Scenario 1 Results ---")
print(f"Stop reason: {result_1.stop_reason}")
print(f"Total steps: {result_1.total_steps}")

{"type":"meta_init","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"{\"format\":\"json\",\"user_query\":\"Please summarize the following text:\\n\\nDocument: War and Peace\\nSource: https://www.gutenberg.org/cache/epub/2600/pg2600.txt\\n\\nWAR AND PEACE\\r\\n\\r\\n\\r\\nBy Leo Tolstoy/Tolstoi\\r\\n\\r\\n\\r\\n    Contents\\r\\n\\r\\n    BOOK ONE: 1805\\r\\n\\r\\n    CHAPTER I\\r\\n\\r\\n    CHAPTER II\\r\\n\\r\\n    CHAPTER III\\r\\n\\r\\n    CHAPTER IV\\r\\n\\r\\n    CHAPTER V\\r\\n\\r\\n    CHAPTER VI\\r\\n\\r\\n    CHAPTER VII\\r\\n\\r\\n    CHAPTER VIII\\r\\n\\r\\n    CHAPTER IX\\r\\n\\r\\n    CHAPTER X\\r\\n\\r\\n    CHAPTER XI\\r\\n\\r\\n    CHAPTER XII\\r\\n\\r\\n    CHAPTER XIII\\r\\n\\r\\n    CHAPTER XIV\\r\\n\\r\\n    CHAPTER XV\\r\\n\\r\\n    CHAPTER XVI\\r\\n\\r\\n    CHAPTER XVII\\r\\n\\r\\n    CHAPTER XVIII\\r\\n\\r\\n    CHAPTER XIX\\r\\n\\r\\n    CHAPTER XX\\r\\n\\r\\n    CHAPTER XXI\\r\\n\\r\\n    CHAPTER XXII\\r\\n\\r\\n    CHAPTER XXIII\\r\\n\\r

2026-03-29T10:57:50.318481Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"I'll read"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" the file to see the full content of your request"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"."}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":""}
{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"path\": \".context/prompt_3cc28ca6-6f4b-4abf-95db-e7429940c855.txt\"}","id":"toolu_01HMKvAPuteo6iewF7gHPoog","name":"read_file"}


2026-03-29T10:57:53.150367Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"I can"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" see this is the beginning of \"War and Peace\" by Leo Tolstoy. Let me continue reading to get more of the actual content beyond"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" the table of contents."}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":""}
{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"path\": \".context/prompt_3cc28ca6-6f4b-4abf-95db-e7429940c855.txt\", \"offset\": 251, \"limit\": 250}","id":"toolu_013iZbUnTRK5k7176SaG1jpo","name":"read_file"}


...

{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"Let me continue reading more sections"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" to get a comprehensive understanding of the text:"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":""}
{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"path\": \".context/prompt_3cc28ca6-6f4b-4abf-95db-e7429940c855.txt\", \"offset\": 1001, \"limit\": 250}","id":"toolu_018ZAGxXTPkdNgsT3weF3U5X","name":"read_file"}


2026-03-29T10:58:05.644261Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"Let me read"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" a bit more to get additional content from the middle and"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" later sections:"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":""}
{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"path\": \".context/prompt_3cc28ca6-6f4b-4abf-95db-e7429940c855.txt\", \"offset\": 1500, \"limit\": 127}","id":"toolu_01C5deGdHWR4HbPPbtkYf9TC","name":"read_file"}


2026-03-29T10:58:09.376840Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"Based"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" on the extensive excerpts I've read from the text, I can now provide you with a comprehensive summary of \""}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"War and Peace\" by Leo Tolstoy.\n\n## Summary of \"War and Peace\" by Leo Tolstoy\n\n**War and Peace** is Leo Tolstoy's epic"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" novel set during the Napoleonic era, spanning from 1805 to 1820. The work is"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" divided into 15 books plus two epilogues and"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" follows the intertwined lives of several Russian aristocratic families against the backdrop of Napoleon"}
{"type":"

In [5]:
print(result_1.final_answer)

Based on the extensive excerpts I've read from the text, I can now provide you with a comprehensive summary of "War and Peace" by Leo Tolstoy.

## Summary of "War and Peace" by Leo Tolstoy

**War and Peace** is Leo Tolstoy's epic novel set during the Napoleonic era, spanning from 1805 to 1820. The work is divided into 15 books plus two epilogues and follows the intertwined lives of several Russian aristocratic families against the backdrop of Napoleon's invasion of Russia.

### Main Elements:

**Setting and Time Period:**
The novel covers the period from 1805 through 1820, following Russia's involvement in the Napoleonic Wars, including the disastrous French invasion of Russia in 1812 and its aftermath.

**Major Characters (introduced in the opening chapters):**
- **Anna Pávlovna Schérer** - A well-connected maid of honor who hosts influential society gatherings in St. Petersburg
- **Prince Vasíli Kurágin** - A high-ranking nobleman with questionable children, including the dissolute A

In [6]:
print(agent_compact.agent_config.last_known_input_tokens)
print(agent_compact.agent_config.last_known_output_tokens)

12106
771


In [7]:
pprint(result_1.cumulative_usage)

Usage(input_tokens=39,
      output_tokens=1541,
      cache_write_tokens=12100,
      cache_read_tokens=28938,
      thinking_tokens=None,
      raw_usage={})


In [8]:
for msg in agent_compact.agent_config.context_messages:
    print(msg.role)
    for content_block in msg.content:
        if content_block.content_block_type == "text":
            content = content_block.text
        elif content_block.content_block_type == "thinking":
            content = content_block.thinking
        else:
            content = content_block.to_dict()
        print(content_block.content_block_type, content)
    print("\n---\n")

Role.USER
ContentBlockType.TEXT Content externalized to .context/prompt_3cc28ca6-6f4b-4abf-95db-e7429940c855.txt. Use read_file with path='.context/prompt_3cc28ca6-6f4b-4abf-95db-e7429940c855.txt' to access the full content.

---

Role.ASSISTANT
ContentBlockType.TEXT I'll read the file to see the full content of your request.
ContentBlockType.TOOL_USE {'content_block_type': 'tool_use', 'tool_name': 'read_file', 'tool_id': 'toolu_01HMKvAPuteo6iewF7gHPoog', 'tool_input': {'path': '.context/prompt_3cc28ca6-6f4b-4abf-95db-e7429940c855.txt'}, 'kwargs': {'caller': {'type': 'direct'}}}

---

Role.USER
ContentBlockType.TOOL_RESULT {'content_block_type': 'tool_result', 'tool_name': 'read_file', 'tool_id': 'toolu_01HMKvAPuteo6iewF7gHPoog', 'tool_result': [{'content_block_type': 'text', 'text': 'Please summarize the following text:\n\nDocument: War and Peace\nSource: https://www.gutenberg.org/cache/epub/2600/pg2600.txt\n\nWAR AND PEACE\n\n\nBy Leo Tolstoy/Tolstoi\n\n\n    Contents\n\n    BOOK ONE

In [9]:
from IPython.lib.display import FileLink
prompt_history = agent_compact.agent_config.conversation_history[0]
sandbox_dir = agent_compact._sandbox.root
expected_prompt_path = f"{sandbox_dir}/.context/prompt_{prompt_history.id}.txt"
FileLink(expected_prompt_path)

/Users/aurosoni/conductor/workspaces/anthropic_agent/bandung/agent_base/providers/anthropic/sandbox_data/11f2b584-6dcc-40d7-8f5d-8a7691dfad10/.context/prompt_3cc28ca6-6f4b-4abf-95db-e7429940c855.txt

## Scenario 2: Large Tool Result

In [10]:
# Scenario 2: Small prompt, but one tool result is too large and should be externalized
# agent_compact = AnthropicAgent(
#     system_prompt=(
#         "You are a research assistant. "
#     ),
#     model="claude-sonnet-4-5",
#     tools=[
#         ReadFileTool().get_tool(),
#         ListDirTreeTool().get_tool(),
#         GlobFileSearchTool().get_tool(),
#     ]+ COMPACTION_TOOLS,
#     compaction_config=STRICT_COMPACTION,
#     externalization_config=STRICT_EXTERNALIZATION,
#     config_adapter=config_adapter,
#     conversation_adapter=conversation_adapter,
#     run_adapter=run_adapter,
# )
# await agent_compact.initialize()

result_2 = await run_with_stream(
    agent_compact,
    "Look up information Leo Tolstoy's War and Peace. Use the large_result_tool to look up "
        "information. After getting results, provide a brief summary."
)
print(f"\n--- Scenario 2 Results ---")
print(f"Stop reason: {result_2.stop_reason}")
print(f"Total steps: {result_2.total_steps}")

{"type":"meta_init","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"format\":\"json\",\"user_query\":\"Look up information Leo Tolstoy's War and Peace. Use the large_result_tool to look up information. After getting results, provide a brief summary.\",\"agent_uuid\":\"11f2b584-6dcc-40d7-8f5d-8a7691dfad10\",\"parent_agent_uuid\":null,\"model\":\"claude-sonnet-4-5\"}"}


2026-03-29T10:58:28.343920Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"query\": \"Leo Tolstoy War and Peace\"}","id":"toolu_01BricpUpvRRS2dSd9BbmCXA","name":"large_result_tool"}


2026-03-29T10:58:31.112893Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"path\": \".context/tool_result_toolu_01BricpUpvRRS2dSd9BbmCXA.txt\"}","id":"toolu_01USYQupeDCCSYN2KJqNJjdP","name":"read_file"}


...

{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"Let"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" me continue reading to get more substantive content:"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":""}
{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"path\": \".context/tool_result_toolu_01BricpUpvRRS2dSd9BbmCXA.txt\", \"offset\": 751, \"limit\": 250}","id":"toolu_01FfWR21Xmvd14EDFf8XdZNz","name":"read_file"}
{"type":"compaction_start","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"reason\":\"threshold\"}"}


2026-03-29T10:58:35.554522Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"##"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" Conversation Summary\n\n**User's Request:**\nThe user asked"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" me to summarize a text that was externalized to a"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" file: `.context/prompt_3cc28ca6-6f4b-4abf-95db-e7429940c855.txt`\n\n**Actions Taken:**\nI"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" read the file in multiple segments using `read_file` with different offsets:\n- Offset "}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"0: Read table of contents showing the novel's structure (15 books, 2 epil"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"ogues, spanning 1805-182

2026-03-29T10:58:44.609249Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"I"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" can see that the large_result_tool has returned the same full text"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" of the novel from Project Gutenberg. This appears to be the actual"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" text of \"War and Peace\" rather than information *about* the work. Let"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" me provide you with a brief summary based on what we know about"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" this literary work:\n\n## Brief Summary\n\nThe lookup returned"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" the full text of **War and Peace** by Leo Tolstoy fr

In [11]:
print(result_2.final_answer)

I can see that the large_result_tool has returned the same full text of the novel from Project Gutenberg. This appears to be the actual text of "War and Peace" rather than information *about* the work. Let me provide you with a brief summary based on what we know about this literary work:

## Brief Summary

The lookup returned the full text of **War and Peace** by Leo Tolstoy from Project Gutenberg rather than analytical information about the work. However, here's what we can summarize about this masterpiece:

**About the Novel:**
- **Author:** Leo Tolstoy (1828-1910)
- **Published:** 1865-1869 (serialized)
- **Length:** One of the longest novels ever written (~1,400 pages)
- **Setting:** Russia during the Napoleonic Wars (1805-1820)

**Structure:**
- 15 Books divided into chapters
- 2 Epilogues (one narrative, one philosophical)
- Spans 15 years of Russian history
- Alternates between "war" scenes (military campaigns) and "peace" scenes (aristocratic society)

**Key Themes:**
- The na

In [12]:
print(agent_compact.agent_config.last_known_input_tokens)
print(agent_compact.agent_config.last_known_output_tokens)

5726
384


In [13]:
pprint(result_2.cumulative_usage)

Usage(input_tokens=58,
      output_tokens=2199,
      cache_write_tokens=20429,
      cache_read_tokens=66203,
      thinking_tokens=None,
      raw_usage={})


In [14]:
for msg in agent_compact.agent_config.context_messages:
    print(msg.role)
    for content_block in msg.content:
        if content_block.content_block_type == "text":
            content = content_block.text
        elif content_block.content_block_type == "thinking":
            content = content_block.thinking
        else:
            content = content_block.to_dict()
        print(content_block.content_block_type, content)
    print("\n---\n")

Role.ASSISTANT
ContentBlockType.TEXT ## Conversation Summary

**User's Request:**
The user asked me to summarize a text that was externalized to a file: `.context/prompt_3cc28ca6-6f4b-4abf-95db-e7429940c855.txt`

**Actions Taken:**
I read the file in multiple segments using `read_file` with different offsets:
- Offset 0: Read table of contents showing the novel's structure (15 books, 2 epilogues, spanning 1805-1820)
- Offsets 251, 501, 751: Continued through table of contents
- Offset 751+: Reached actual content starting with Book One: 1805, Chapter I
- Read several opening chapters introducing characters and setting

**Document Details:**
- **Title:** War and Peace by Leo Tolstoy
- **Source:** Project Gutenberg (https://www.gutenberg.org/cache/epub/2600/pg2600.txt)
- **Content:** Full text of the novel with extensive table of contents and narrative text

**Summary Provided:**
I delivered a comprehensive summary covering:
- Setting and time period (1805-1820, Napoleonic Wars)
- Major 

In [15]:
context_tool_result_msg_2 = first_tool_result_message(agent_compact.agent_config.context_messages)

context_block_2 = next(
    block for block in context_tool_result_msg_2.content
    if isinstance(block, ToolResultContent)
)
sandbox_dir = agent_compact._sandbox.root
expected_tool_result_path_2 = f"{sandbox_dir}/.context/tool_result_{context_block_2.tool_id}.txt"
FileLink(expected_tool_result_path_2)

/Users/aurosoni/conductor/workspaces/anthropic_agent/bandung/agent_base/providers/anthropic/sandbox_data/11f2b584-6dcc-40d7-8f5d-8a7691dfad10/.context/tool_result_toolu_01BricpUpvRRS2dSd9BbmCXA.txt

## Scenario 3: Parallel Tool Calls overflow context

In [16]:
# Scenario 3: Parallel tool calls are individually fine but overflow together
# agent_compact = AnthropicAgent(
#     system_prompt=(
#         "You are a research assistant. When asked to compare multiple topics, "
#         "ALWAYS use the medium_result_tool for EACH topic IN PARALLEL "
#         "(use all tool calls at once, not sequentially). "
#         "After getting all results, write a brief comparison."
#     ),
#     model="claude-sonnet-4-5",
#     tools=[
#         ReadFileTool().get_tool(),
#         ListDirTreeTool().get_tool(),
#         GlobFileSearchTool().get_tool(),
#     ]+ COMPACTION_TOOLS,
#     compaction_config=STRICT_COMPACTION,
#     externalization_config=STRICT_EXTERNALIZATION,
#     config_adapter=config_adapter,
#     conversation_adapter=conversation_adapter,
#     run_adapter=run_adapter,
# )
# await agent_compact.initialize()

result_3 = await run_with_stream(
    agent_compact,
    "Compare these three topics: leo tolstoy's war and peace, the index of the book, and the first chapter of the book."
    "Look up each one using the medium_result_tool. ALWAYS use the medium_result_tool for EACH topic IN PARALLEL."
)
print(f"\n--- Scenario 3 Results ---")
print(f"Stop reason: {result_3.stop_reason}")
print(f"Total steps: {result_3.total_steps}")

{"type":"meta_init","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"format\":\"json\",\"user_query\":\"Compare these three topics: leo tolstoy's war and peace, the index of the book, and the first chapter of the book.Look up each one using the medium_result_tool. ALWAYS use the medium_result_tool for EACH topic IN PARALLEL.\",\"agent_uuid\":\"11f2b584-6dcc-40d7-8f5d-8a7691dfad10\",\"parent_agent_uuid\":null,\"model\":\"claude-sonnet-4-5\"}"}
{"type":"compaction_start","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"reason\":\"threshold\"}"}


2026-03-29T10:58:54.011527Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"##"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" Conversation History Summary\n\n**User's Goal:**\nSummarize a text that was externalized to file `.context/prompt_3cc28ca6-6f4b-"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"4abf-95db-e7429940c855.txt`\n\n**Actions Completed:**\n- Read the externalized file using multiple"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" `read_file` calls with different offsets (0, 251, 501, 751, and beyond)\n- File"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" contained: Full text of \"War and Peace\" by Leo Tolstoy from Project Gutenberg (https://www.gutenberg.org"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"/cache/epub/2600/pg2600.txt)\n- File structure: Table of co

2026-03-29T10:58:59.653973Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"topic\": \"leo tolstoy's war and peace\"}","id":"toolu_01AJzBghzFcdnipdH5wJsgEh","name":"medium_result_tool"}
{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"topic\": \"the index of the book\"}","id":"toolu_016fAipVV3RoK4iYRyiuqcwx","name":"medium_result_tool"}
{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"topic\": \"the first chapter of the book\"}","id":"toolu_01W3gpcFnGmcSjnqqzA4pHf6","name":"medium_result_tool"}
{"type":"compaction_start","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"reason\":\"threshold\"}"}


...

{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"path\": \".context/tool_result_toolu_01AJzBghzFcdnipdH5wJsgEh.txt\"}","id":"toolu_01TdRzWCwa1Ehv5ZAMnTetvp","name":"read_file"}
{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"path\": \".context/tool_result_toolu_016fAipVV3RoK4iYRyiuqcwx.txt\"}","id":"toolu_01JZTcYWdbvnghq7Vy9ApsmD","name":"read_file"}
{"type":"tool_call","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"path\": \".context/tool_result_toolu_01W3gpcFnGmcSjnqqzA4pHf6.txt\"}","id":"toolu_016QBet2HeofeX6WYVVWsFSP","name":"read_file"}
{"type":"compaction_start","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":true,"delta":"{\"reason\":\"threshold\"}"}


2026-03-29T10:59:11.845815Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"#"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" Conversation History Summary\n\n## User's Goals\n1. Initially: Summarize a text"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" file that was externalized\n2. Currently: Look up information about Leo"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" Tolstoy's \"War and Peace\" and provide a brief summary\n\n## Key Actions & Results\n\n### First"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" Request (File Summary)\n- **File:** `.context/prompt_3"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"cc28ca6-6f4b-4ab"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"f-95db-e7429940c855"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8

2026-03-29T10:59:23.025596Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"##"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" Comparison of the Three Topic Lookups\n\nI've looked"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" up all three topics in parallel using the `medium_result_"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":"tool`. Here's what I found:\n\n### **Key Finding: All"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" Three Queries Returned Identical Results**\n\nSurprisingly, all three searches returned **exactly"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" the same content** - the beginning of the full text of \"War and Peace\" from"}
{"type":"text","agent":"11f2b584-6dcc-40d7-8f5d-8a7691dfad10","final":false,"delta":" Project Gutenberg, starting with the table o

In [17]:
print(result_3.final_answer)

## Comparison of the Three Topic Lookups

I've looked up all three topics in parallel using the `medium_result_tool`. Here's what I found:

### **Key Finding: All Three Queries Returned Identical Results**

Surprisingly, all three searches returned **exactly the same content** - the beginning of the full text of "War and Peace" from Project Gutenberg, starting with the table of contents.

### Detailed Comparison:

1. **Topic: "leo tolstoy's war and peace"**
   - Returned: Full document beginning with table of contents
   - Shows structure: 15 Books spanning 1805-1813, plus epilogues

2. **Topic: "the index of the book"**
   - Returned: **Identical content** to topic #1
   - Same table of contents listing all chapters
   
3. **Topic: "the first chapter of the book"**
   - Returned: **Identical content** to topics #1 and #2
   - Also shows table of contents, not the actual first chapter text

### Observations:

- **No differentiation**: The tool appears to be returning the same source do

In [18]:
print(agent_compact.agent_config.last_known_input_tokens)
print(agent_compact.agent_config.last_known_output_tokens)

5014
410


In [19]:
pprint(result_3.cumulative_usage)

Usage(input_tokens=75,
      output_tokens=2974,
      cache_write_tokens=38078,
      cache_read_tokens=66203,
      thinking_tokens=None,
      raw_usage={})


In [20]:
for msg in agent_compact.agent_config.context_messages:
    print(msg.role)
    for content_block in msg.content:
        if content_block.content_block_type == "text":
            content = content_block.text
        elif content_block.content_block_type == "thinking":
            content = content_block.thinking
        else:
            content = content_block.to_dict()
        print(content_block.content_block_type, content)
    print("\n---\n")


Role.ASSISTANT
ContentBlockType.TEXT # Conversation History Summary

## User's Goals
1. Initially: Summarize a text file that was externalized
2. Currently: Look up information about Leo Tolstoy's "War and Peace" and provide a brief summary

## Key Actions & Results

### First Request (File Summary)
- **File:** `.context/prompt_3cc28ca6-6f4b-4abf-95db-e7429940c855.txt`
- **Contents:** Complete text of "War and Peace" by Leo Tolstoy from Project Gutenberg (https://www.gutenberg.org/cache/epub/2600/pg2600.txt)
- **Structure:** 15 books + 2 epilogues, spanning 1805-1820
- **Summary provided:** Novel set in Russia during Napoleonic Wars, featuring main characters Anna Pávlovna, Prince Vasíli, Pierre Bezúkhov, Princess Hélène, Prince Andrew Bolkónski
- **Status:** Completed successfully

### Second Request (Information Lookup)
- **Tool used:** `large_result_tool` with query "Leo Tolstoy War and Peace"
- **Result file:** `.context/tool_result_toolu_01BricpUpvRRS2dSd9BbmCXA.txt`
- **Issue:** 

In [21]:
sandbox_dir = agent_compact._sandbox.root
print(sandbox_dir)

/Users/aurosoni/conductor/workspaces/anthropic_agent/bandung/agent_base/providers/anthropic/sandbox_data/11f2b584-6dcc-40d7-8f5d-8a7691dfad10


# Steer and Abort

## Streaming Abort and Steer

In [3]:
from agent_base.providers.anthropic import AnthropicAgent, AnthropicLLMConfig
from agent_base.tools import tool

agent = AnthropicAgent(
        system_prompt=(
            "You are a helpful assistant. Think carefully and answer in a detailed, "
            "step-by-step way."
        ),
        model="claude-sonnet-4-5",
        config=AnthropicLLMConfig(thinking_tokens=1024, max_tokens=4096),
    )

In [ ]:
prompt = "Explain in a very detailed way how to solve 5 + (10 * 2). \
    Think first, then explain every intermediate step slowly."

demo = await start_demo(agent, prompt)

Run started.
Now manually trigger one of these in a later cell:
  await abort_now()
  await steer_now('your new instruction')
  await wait_now()


{"type":"meta_init","agent":"52a3f417-3820-48b1-82e8-a01366e963fe","final":true,"delta":"{\"format\":\"json\",\"user_query\":\"Explain in a very detailed way how to solve 5 + (10 * 2).     Think first, then explain every intermediate step slowly.\",\"agent_uuid\":\"52a3f417-3820-48b1-82e8-a01366e963fe\",\"parent_agent_uuid\":null,\"model\":\"claude-sonnet-4-5\"}"}


2026-03-22T04:41:58.810392Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"thinking","agent":"52a3f417-3820-48b1-82e8-a01366e963fe","final":false,"delta":"The user wants"}
{"type":"thinking","agent":"52a3f417-3820-48b1-82e8-a01366e963fe","final":false,"delta":" me to solve"}
{"type":"thinking","agent":"52a3f417-3820-48b1-82e8-a01366e963fe","final":false,"delta":" 5 + (10 *"}
{"type":"thinking","agent":"52a3f417-3820-48b1-82e8-a01366e963fe","final":false,"delta":" 2) and explain it"}
{"type":"thinking","agent":"52a3f417-3820-48b1-82e8-a01366e963fe","final":false,"delta":" in detail"}
{"type":"thinking","agent":"52a3f417-3820-48b1-82e8-a01366e963fe","final":false,"delta":" with"}
{"type":"thinking","agent":"52a3f417-3820-48b1-82e8-a01366e963fe","final":false,"delta":" intermediate steps.\n\nLet me think"}
{"type":"thinking","agent":"52a3f417-3820-48b1-82e8-a01366e963fe","final":false,"delta":" about this:\n- I"}
{"type":"thinking","agent":"52a3f417-3820-48b1-82e8-a01366e963fe","final":false,"delta":" need to follow"}
{"type":"thinking","agent":"52a3f41

In [5]:
res = await demo.abort()

stop_reason=aborted
was_aborted=True
abort_phase=None
total_steps=0
final_answer=''


In [12]:
agent.agent_config.context_messages

[Message(id='9a25f9d4-b3c1-44f7-9ade-f9b8eb60428d', role=<Role.USER: 'user'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='Explain in a very detailed way how to solve 5 + (10 * 2).     Think first, then explain every intermediate step slowly.')], stop_reason=None, usage=None, provider='', model='', usage_kwargs={}),
 Message(id='306b41a1-59f7-4d26-a189-b44283693c11', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking="The user wants me to solve 5 + (10 * 2) and explain it in detail with intermediate steps.\n\nLet me think about this:\n- I need to follow the order of operations (PEMDAS/BODMAS)\n- Parentheses/Brackets first\n- Within the parentheses, there's multiplication\n- Then addition outside the parentheses\n\nLet me work through this step by step.", signature='EvEDCkYICxgCKkDbnr28Y+1F6yI1QSy3UEXYuJqkztyHh2J+ZENgr0+TZNVUnIDSGA3djLujnI1c/nrTJvlGjbgP6

In [13]:
res2 = await agent.run("Explain the BODMAS Rule first, then solve the expression.")

2026-03-22T04:48:37.489206Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


In [15]:
agent.agent_config.context_messages

[Message(id='9a25f9d4-b3c1-44f7-9ade-f9b8eb60428d', role=<Role.USER: 'user'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='Explain in a very detailed way how to solve 5 + (10 * 2).     Think first, then explain every intermediate step slowly.')], stop_reason=None, usage=None, provider='', model='', usage_kwargs={}),
 Message(id='306b41a1-59f7-4d26-a189-b44283693c11', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking="The user wants me to solve 5 + (10 * 2) and explain it in detail with intermediate steps.\n\nLet me think about this:\n- I need to follow the order of operations (PEMDAS/BODMAS)\n- Parentheses/Brackets first\n- Within the parentheses, there's multiplication\n- Then addition outside the parentheses\n\nLet me work through this step by step.", signature='EvEDCkYICxgCKkDbnr28Y+1F6yI1QSy3UEXYuJqkztyHh2J+ZENgr0+TZNVUnIDSGA3djLujnI1c/nrTJvlGjbgP6

In [17]:
agent.agent_config.conversation_history

[Message(id='9a25f9d4-b3c1-44f7-9ade-f9b8eb60428d', role=<Role.USER: 'user'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='Explain in a very detailed way how to solve 5 + (10 * 2).     Think first, then explain every intermediate step slowly.')], stop_reason=None, usage=None, provider='', model='', usage_kwargs={}),
 Message(id='306b41a1-59f7-4d26-a189-b44283693c11', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking="The user wants me to solve 5 + (10 * 2) and explain it in detail with intermediate steps.\n\nLet me think about this:\n- I need to follow the order of operations (PEMDAS/BODMAS)\n- Parentheses/Brackets first\n- Within the parentheses, there's multiplication\n- Then addition outside the parentheses\n\nLet me work through this step by step.", signature='EvEDCkYICxgCKkDbnr28Y+1F6yI1QSy3UEXYuJqkztyHh2J+ZENgr0+TZNVUnIDSGA3djLujnI1c/nrTJvlGjbgP6

In [ ]:
# demo = await start_demo(agent, prompt)
await demo.steer("Explain the BODMAS rule first, then solve the expression.")

2026-03-22T04:37:29.000004Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


stop_reason=end_turn
was_aborted=False
abort_phase=None
total_steps=1
final_answer="# BODMAS Rule Explained\n\n**BODMAS** is an acronym that tells us the correct order to perform mathematical operations in an expression. Let me break down what each letter means:\n\n## The Order of Operations:\n\n1. **B - Brackets** (also called Parentheses)\n   - Operations inside brackets ( ), { }, [ ] are done first\n   - Innermost brackets are solved before outer ones\n\n2. **O - Orders** (also called Powers, Indices, or Exponents)\n   - This includes powers (x²), square roots (√), cube roots, etc.\n\n3. **D - Division** \n   - Division operations (÷ or /)\n\n4. **M - Multiplication**\n   - Multiplication operations (× or *)\n   - *Note: Division and Multiplication have equal priority and are done left to right*\n\n5. **A - Addition**\n   - Addition operations (+)\n\n6. **S - Subtraction**\n   - Subtraction operations (-)\n   - *Note: Addition and Subtraction have equal priority and are done left to

AgentResult(final_message=Message(id='45be24e9-31d9-418f-af14-250f1a57a716', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking='The user wants me to first explain the BODMAS rule, and then solve the expression 5 + (10 * 2) using that rule. I should be thorough and detailed.', signature='ErsCCkYICxgCKkBQy5+1/fEfrUoeqMWTnAkW/Q6aE4/G9yR1NzS6oPtjt4JfWtrBPh5vKBK1w7JGMv8TFz1LqLBfeFNHxZodkgZ7Egw7gZEqWUoPbJ3OH8QaDN9uC5l1k15Z0rz+vSIwhdWoJFAFLdNFEPa4S5e3aT53/qfuwax7Y9TPLFI0AicDg4G0CMNGVPNrupKGD1gWKqIBnl7jNt3d/yPegLOeXWH65a/UHal+mxBdz9LQx8xKH7Hmsl5bDNoffRcu3hGkUKJOTGND5ejc3/7BhLQ4YRbFKQtWw2vyHdkHWzphpCuZkm3ofvoavJsdCiOYvqg7MwiFzzpDjK0z7kWlJhR7hVZO8B8XB37z49xwHWCtNc7y7O1oK3eKrVyuJ66KtBCAS5OMAzL+vB2eqt5CiVWOnuRBQlqAGAE='), TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text="# BODMAS Rule Explained\n\n**BODMAS** is an acronym that tells us the correct order to perform mathematical 

In [ ]:
# demo = await start_demo(agent, prompt)
res3 = await demo.steer("Give only a very concise explaination.")

2026-03-22T04:52:17.486213Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


stop_reason=end_turn
was_aborted=False
abort_phase=None
total_steps=2
final_answer='# BODMAS Rule (Concise)\n\n**BODMAS** = Order of operations:\n- **B**rackets\n- **O**rders (powers/roots)\n- **D**ivision & **M**ultiplication (left to right)\n- **A**ddition & **S**ubtraction (left to right)\n\n---\n\n# Solve: 5 + (10 * 2)\n\n**Step 1:** Brackets first → 10 * 2 = 20\n\n**Step 2:** Addition → 5 + 20 = 25\n\n**Answer: 25**'


In [19]:
agent.agent_config.context_messages

[Message(id='9a25f9d4-b3c1-44f7-9ade-f9b8eb60428d', role=<Role.USER: 'user'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='Explain in a very detailed way how to solve 5 + (10 * 2).     Think first, then explain every intermediate step slowly.')], stop_reason=None, usage=None, provider='', model='', usage_kwargs={}),
 Message(id='306b41a1-59f7-4d26-a189-b44283693c11', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking="The user wants me to solve 5 + (10 * 2) and explain it in detail with intermediate steps.\n\nLet me think about this:\n- I need to follow the order of operations (PEMDAS/BODMAS)\n- Parentheses/Brackets first\n- Within the parentheses, there's multiplication\n- Then addition outside the parentheses\n\nLet me work through this step by step.", signature='EvEDCkYICxgCKkDbnr28Y+1F6yI1QSy3UEXYuJqkztyHh2J+ZENgr0+TZNVUnIDSGA3djLujnI1c/nrTJvlGjbgP6

In [21]:
print(res3.final_answer)

# BODMAS Rule (Concise)

**BODMAS** = Order of operations:
- **B**rackets
- **O**rders (powers/roots)
- **D**ivision & **M**ultiplication (left to right)
- **A**ddition & **S**ubtraction (left to right)

---

# Solve: 5 + (10 * 2)

**Step 1:** Brackets first → 10 * 2 = 20

**Step 2:** Addition → 5 + 20 = 25

**Answer: 25**


## Tool Call Abort and Steer

In [3]:
from agent_base.providers.anthropic import AnthropicAgent, AnthropicLLMConfig
from agent_base.tools import tool

@tool
async def slow_calculate(expression: str, delay_seconds: int = 10) -> str:
    """Evaluate a mathematical expression after a short delay."""
    await asyncio.sleep(delay_seconds)
    return str(eval(expression, {"__builtins__": {}}, {}))

agent = AnthropicAgent(
        system_prompt=(
            "You are a helpful assistant. Always use the slow_calculate tool for "
            "math questions before answering."
        ),
        model="claude-sonnet-4-5",
        config=AnthropicLLMConfig(thinking_tokens=1024, max_tokens=4096),
        tools=[slow_calculate],
    )

In [ ]:
demo = await start_demo(agent, "Use slow_calculate to compute 25 * 4 with delay_seconds=10, "
            "then explain the result in a few sentences.")

{"type":"meta_init","agent":"3406d1fa-da5f-4965-aac9-ab2e7a582c7b","final":true,"delta":"{\"format\":\"json\",\"user_query\":\"Use slow_calculate to compute 25 * 4 with delay_seconds=10, then explain the result in a few sentences.\",\"agent_uuid\":\"3406d1fa-da5f-4965-aac9-ab2e7a582c7b\",\"parent_agent_uuid\":null,\"model\":\"claude-sonnet-4-5\"}"}


2026-03-29T07:31:26.158348Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"thinking","agent":"3406d1fa-da5f-4965-aac9-ab2e7a582c7b","final":false,"delta":"The user wants me to use the slow"}
{"type":"thinking","agent":"3406d1fa-da5f-4965-aac9-ab2e7a582c7b","final":false,"delta":"_calculate function to compute 25 * 4 with a delay of 10 seconds. I have all the required parameters:\n- expression: \"25 * 4\"\n- delay_seconds: 10\n\nLet"}
{"type":"thinking","agent":"3406d1fa-da5f-4965-aac9-ab2e7a582c7b","final":false,"delta":" me make this function call."}
{"type":"thinking","agent":"3406d1fa-da5f-4965-aac9-ab2e7a582c7b","final":true,"delta":""}
{"type":"tool_call","agent":"3406d1fa-da5f-4965-aac9-ab2e7a582c7b","final":true,"delta":"{\"expression\": \"25 * 4\", \"delay_seconds\": 10}","id":"toolu_01SJduWxyH1cmpqW7zXBw5wr","name":"slow_calculate"}


In [5]:
res1 = await demo.abort()

stop_reason=aborted
was_aborted=True
abort_phase=None
total_steps=1
final_answer=''


In [6]:
agent.agent_config.context_messages

[Message(id='f8f3f154-676a-45e9-a29a-3d4cb38de7f7', role=<Role.USER: 'user'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='Use slow_calculate to compute 25 * 4 with delay_seconds=10, then explain the result in a few sentences.')], stop_reason=None, usage=None, provider='', model='', usage_kwargs={}),
 Message(id='9de29a9b-4083-4fcf-817d-3996466c7adb', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking='The user wants me to use the slow_calculate function to compute 25 * 4 with a delay of 10 seconds. I have all the required parameters:\n- expression: "25 * 4"\n- delay_seconds: 10\n\nLet me make this function call.', signature='EpkDCmQIDBgCKkAmQ2/XPPRMzMhfMnF4sQ3WWYKgTBCvTHwlBkRL8yuEgjN0jQCt7HhbD+U2FECGy2pL50m/9pHbtnclMRU6zNgcMhpjbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOTgAEgxr5PtODhefsAaJOlIaDLcfRifBIRCmlCzMxCIwv4F9MkHq2WOpemE6Tx1ymDAHspjMgcz5m0ZKtoub9MkyW1yThxL

In [7]:
res2 = await agent.run("Tell me why you stopped then continue your work. Call the slow calculate again.")
res2

2026-03-29T07:31:52.010694Z [warning  ] defensive_sanitize_repaired_chain [agent_base.providers.anthropic.provider] original_count=4 sanitized_count=3
2026-03-29T07:31:57.817841Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-29T07:32:11.040490Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


AgentResult(final_message=Message(id='1f43cf24-4a55-4688-bd33-cd77423ede56', role=<Role.ASSISTANT: 'assistant'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='Perfect! The calculation completed successfully. \n\n**Result: 25 × 4 = 100**\n\nThis is a straightforward multiplication problem. Twenty-five multiplied by four equals one hundred. This makes intuitive sense because 25 is one-quarter of 100, so four quarters make a whole. This relationship is commonly used in monetary calculations (four quarters equal one dollar) and percentage conversions (25% times 4 equals 100%).')], stop_reason='end_turn', usage=Usage(input_tokens=6, output_tokens=100, cache_write_tokens=1037, cache_read_tokens=0, thinking_tokens=None, raw_usage={'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 1037}, 'cache_creation_input_tokens': 1037, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 6, 'output_tokens'

In [32]:
res2 = await run_with_stream(agent, "Continue your work. Call the slow calculate again.")
_print_result(res2)
res2

{"type":"meta_init","agent":"8065346a-cd92-4842-86b9-98487bb5a159","final":true,"delta":"{\"format\":\"json\",\"user_query\":\"Continue your work. Call the slow calculate again.\",\"agent_uuid\":\"8065346a-cd92-4842-86b9-98487bb5a159\",\"parent_agent_uuid\":null,\"model\":\"claude-sonnet-4-5\"}"}


2026-03-29T07:26:23.272791Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"thinking","agent":"8065346a-cd92-4842-86b9-98487bb5a159","final":false,"delta":"The user is asking me to call"}
{"type":"thinking","agent":"8065346a-cd92-4842-86b9-98487bb5a159","final":false,"delta":" slow_calculate again, but they haven't specified what calculation to perform. They just said to"}
{"type":"thinking","agent":"8065346a-cd92-4842-86b9-98487bb5a159","final":false,"delta":" \"continue your work\" and \"call the slow calculate again.\"\n\nI should ask what calculation they'd like me to perform, since I need an expression to calculate. I"}
{"type":"thinking","agent":"8065346a-cd92-4842-86b9-98487bb5a159","final":false,"delta":" shouldn't just make up a calculation without knowing what they want."}
{"type":"thinking","agent":"8065346a-cd92-4842-86b9-98487bb5a159","final":true,"delta":""}
{"type":"text","agent":"8065346a-cd92-4842-86b9-98487bb5a159","final":false,"delta":"I'd be happy to call slow"}
{"type":"text","agent":"8065346a-cd92-4842-86b9-98487bb5a159","final"

AgentResult(final_message=Message(id='c53bc6c2-6a88-4ba8-b20b-b60f9555154c', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking='The user is asking me to call slow_calculate again, but they haven\'t specified what calculation to perform. They just said to "continue your work" and "call the slow calculate again."\n\nI should ask what calculation they\'d like me to perform, since I need an expression to calculate. I shouldn\'t just make up a calculation without knowing what they want.', signature='EqgECmQIDBgCKkDKe57goQUu7AFtDSw6dB8isCtjFFMxAx44JQKfh4hw1pKfL8p3Enp0YNr8vd7tIpwsqAN5VNuf63ZUZ2C5sS6QMhpjbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOTgAEgx5gysm55JVEPAvImsaDFENusFnQvdZ6+8hRyIwDbmyAM4D2GHij7Luih+sLZpMSeuY4caNTPMKoCioZXA28kFU0x9pCvMkbK6tvBRqKvEC/LZfbrPG7NfOHlLElCeBU6CPsF1aoeIm0m1WJUjpIs3Sfv+FbDNV+sfkaV8Ay5fZvtQOdxINnSBG9QjJ+Q07drX/LoXSNQiigbPambeRhP2gcMembOsJGT/GRhXDbL0CStNcxb8YTLp80fV+BRP5myIzvZl2i/uaX

In [8]:
for msg in agent.agent_config.context_messages:
    print(msg.role)
    for content_block in msg.content:
        if content_block.content_block_type == "text":
            content = content_block.text
        elif content_block.content_block_type == "thinking":
            content = content_block.thinking
        else:
            content = content_block.to_dict()
        print(content_block.content_block_type, content)
    print("\n---\n")

Role.USER
ContentBlockType.TEXT Use slow_calculate to compute 25 * 4 with delay_seconds=10, then explain the result in a few sentences.

---

Role.ASSISTANT
ContentBlockType.THINKING The user wants me to use the slow_calculate function to compute 25 * 4 with a delay of 10 seconds. I have all the required parameters:
- expression: "25 * 4"
- delay_seconds: 10

Let me make this function call.
ContentBlockType.TOOL_USE {'content_block_type': 'tool_use', 'tool_name': 'slow_calculate', 'tool_id': 'toolu_01SJduWxyH1cmpqW7zXBw5wr', 'tool_input': {'expression': '25 * 4', 'delay_seconds': 10}, 'kwargs': {'caller': {'type': 'direct'}}}

---

Role.USER
ContentBlockType.TOOL_RESULT {'content_block_type': 'tool_result', 'tool_name': 'slow_calculate', 'tool_id': 'toolu_01SJduWxyH1cmpqW7zXBw5wr', 'tool_result': [{'content_block_type': 'text', 'text': 'Error: Tool execution was aborted by the user.', 'kwargs': {}}], 'is_error': True, 'kwargs': {}}
ContentBlockType.TEXT Tell me why you stopped then con

In [9]:
res3 = await agent.run("Now calculate your previous result into 5. Call the slow calculate again.")
res3

2026-03-22T06:05:37.583351Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-22T06:05:50.345909Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


AgentResult(final_message=Message(id='f3cbb87f-36b1-40ec-a33c-263757ccee6d', role=<Role.ASSISTANT: 'assistant'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='The calculation is complete! Dividing 100 by 5 gives us **20**.\n\nThis makes perfect sense: if you take 100 and divide it into 5 equal parts, each part contains 20. You can verify this by multiplying back: 20 × 5 = 100. This is another useful mathematical relationship to remember - 100 divided by 5 equals 20, which means 5 "goes into" 100 exactly 20 times with no remainder.')], stop_reason='end_turn', usage=Usage(input_tokens=6, output_tokens=113, cache_write_tokens=1259, cache_read_tokens=0, thinking_tokens=None, raw_usage={'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 1259}, 'cache_creation_input_tokens': 1259, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 6, 'output_tokens': 113, 'service_tier': 'standard'}), provid

In [10]:
print(res3.final_answer)

The calculation is complete! Dividing 100 by 5 gives us **20**.

This makes perfect sense: if you take 100 and divide it into 5 equal parts, each part contains 20. You can verify this by multiplying back: 20 × 5 = 100. This is another useful mathematical relationship to remember - 100 divided by 5 equals 20, which means 5 "goes into" 100 exactly 20 times with no remainder.


## Relay Abort and Steer

In [9]:
from agent_base.providers.anthropic import AnthropicAgent, AnthropicLLMConfig
from agent_base.tools import tool

@tool
def quick_calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    return str(eval(expression, {"__builtins__": {}}, {}))


@tool(executor="frontend")
def manual_confirm(message: str) -> str:
    """Ask the user for confirmation before continuing."""
    return f"Confirmation requested: {message}"


agent = AnthropicAgent(
    system_prompt=(
        "You are a helpful assistant. Use quick_calculate for math. If the result "
        "is greater than 50, ask for confirmation with manual_confirm before "
        "giving the final answer."
    ),
    model="claude-sonnet-4-5",
    config=AnthropicLLMConfig(thinking_tokens=1024, max_tokens=4096),
    tools=[quick_calculate],
    frontend_tools=[manual_confirm],
)


In [ ]:
prompt = "Calculate 25 * 4 for me."
demo = await start_demo(agent, prompt)

{"type":"meta_init","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":true,"delta":"{\"format\":\"json\",\"user_query\":\"Calculate 25 * 4 for me.\",\"agent_uuid\":\"80ee038b-ce04-40e3-a577-f4c203dc4d37\",\"parent_agent_uuid\":null,\"model\":\"claude-sonnet-4-5\"}"}


2026-03-29T07:32:50.316009Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"thinking","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":false,"delta":"The user wants me to calculate 25"}
{"type":"thinking","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":false,"delta":" * 4. I should use the quick_calculate function for this.\n\nLet me calculate: 25 * 4 "}
{"type":"thinking","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":false,"delta":"= 100\n\nSince the result is 100, which is greater than 50, I need to ask for confirmation before giving"}
{"type":"thinking","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":false,"delta":" the final answer.\n\nLet me first calculate it, then check if I need confirmation."}
{"type":"thinking","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":true,"delta":""}
{"type":"tool_call","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":true,"delta":"{\"expression\": \"25 * 4\"}","id":"toolu_01VsgsnshqESPxysS8NADFgt","name":"quick_calculate"}


2026-03-29T07:32:53.345732Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{"type":"text","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":false,"delta":"Since"}
{"type":"text","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":false,"delta":" the result is 100, which is greater than 50, I need to confirm before providing the final answer."}
{"type":"text","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":true,"delta":""}
{"type":"tool_call","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":true,"delta":"{\"message\": \"The result is 100. Do you want me to provide this answer?\"}","id":"toolu_013tkwTA7mEpVvipbwe6SF2E","name":"manual_confirm"}
{"type":"awaiting_frontend_tools","agent":"80ee038b-ce04-40e3-a577-f4c203dc4d37","final":true,"delta":"{\"tools\":[{\"tool_use_id\":\"toolu_013tkwTA7mEpVvipbwe6SF2E\",\"name\":\"manual_confirm\",\"input\":{\"message\":\"The result is 100. Do you want me to provide this answer?\"}}]}"}


In [11]:
res1 = await demo.abort()
res1

stop_reason=aborted
was_aborted=True
abort_phase=None
total_steps=2
final_answer='Since the result is 100, which is greater than 50, I need to confirm before providing the final answer.'


AgentResult(final_message=Message(id='a0c223e0-f84c-4ed1-8bd6-cad4ff6b60d8', role=<Role.ASSISTANT: 'assistant'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='Since the result is 100, which is greater than 50, I need to confirm before providing the final answer.'), ToolUseContent(content_block_type=<ContentBlockType.TOOL_USE: 'tool_use'>, kwargs={'caller': {'type': 'direct'}}, tool_name='manual_confirm', tool_id='toolu_013tkwTA7mEpVvipbwe6SF2E', tool_input={'message': 'The result is 100. Do you want me to provide this answer?'})], stop_reason='tool_use', usage=Usage(input_tokens=864, output_tokens=92, cache_write_tokens=0, cache_read_tokens=0, thinking_tokens=None, raw_usage={'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 864, 'output_tokens': 92, 'service_tier': 'standard'}), provider='anthrop

In [12]:
agent.agent_config.context_messages

[Message(id='348b3bf9-fc33-43c9-9b39-1f09d0b795b4', role=<Role.USER: 'user'>, content=[TextContent(content_block_type=<ContentBlockType.TEXT: 'text'>, kwargs={}, text='Calculate 25 * 4 for me.')], stop_reason=None, usage=None, provider='', model='', usage_kwargs={}),
 Message(id='9dd20405-8cdb-4fac-93b9-dac3640b18aa', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking='The user wants me to calculate 25 * 4. I should use the quick_calculate function for this.\n\nLet me calculate: 25 * 4 = 100\n\nSince the result is 100, which is greater than 50, I need to ask for confirmation before giving the final answer.\n\nLet me first calculate it, then check if I need confirmation.', signature='EvMDCmQIDBgCKkDCpE82U1/UXGaTbkHz4vPw6ZAHwtJ1dS9UNkMcrG+9fsTy6tCtuD8JBWtIhTRoDsj9fYQOKTIbx052O+EPB5TUMhpjbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOTgAEgxuSmNpe0hDsjuJne8aDO4scoO6y4UkBTsByiIwHQ7AJ3Tz/qXngj+H2I1eJ7VznaOwboIBXiz3QH

In [13]:
for msg in agent.agent_config.context_messages:
    print(msg.role)
    for content_block in msg.content:
        if content_block.content_block_type == "text":
            content = content_block.text
        elif content_block.content_block_type == "thinking":
            content = content_block.thinking
        else:
            content = content_block.to_dict()
        print(content_block.content_block_type, content)
    print("\n---\n")

Role.USER
ContentBlockType.TEXT Calculate 25 * 4 for me.

---

Role.ASSISTANT
ContentBlockType.THINKING The user wants me to calculate 25 * 4. I should use the quick_calculate function for this.

Let me calculate: 25 * 4 = 100

Since the result is 100, which is greater than 50, I need to ask for confirmation before giving the final answer.

Let me first calculate it, then check if I need confirmation.
ContentBlockType.TOOL_USE {'content_block_type': 'tool_use', 'tool_name': 'quick_calculate', 'tool_id': 'toolu_01VsgsnshqESPxysS8NADFgt', 'tool_input': {'expression': '25 * 4'}, 'kwargs': {'caller': {'type': 'direct'}}}

---

Role.USER
ContentBlockType.TOOL_RESULT {'content_block_type': 'tool_result', 'tool_name': 'quick_calculate', 'tool_id': 'toolu_01VsgsnshqESPxysS8NADFgt', 'tool_result': [{'content_block_type': 'text', 'text': '100', 'kwargs': {}}], 'is_error': False, 'kwargs': {}}

---

Role.ASSISTANT
ContentBlockType.TEXT Since the result is 100, which is greater than 50, I need to 

In [14]:
res2 = await agent.run("First tell me what made you stop. Then regardless you have my permission to execute any calculations, then tell me the result.")
res2

2026-03-29T07:33:04.237067Z [warning  ] defensive_sanitize_repaired_message [agent_base.providers.anthropic.provider] role=user
2026-03-29T07:33:08.956470Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


AgentResult(final_message=Message(id='7df6791f-a8b2-4628-93ed-b698a2e619f8', role=<Role.ASSISTANT: 'assistant'>, content=[ThinkingContent(content_block_type=<ContentBlockType.THINKING: 'thinking'>, kwargs={}, thinking='The user is asking two things:\n1. Why did I stop?\n2. They\'re giving me permission to execute calculations, and want the result\n\nLet me explain: I stopped because according to my instructions, "If the result is greater than 50, ask for confirmation with manual_confirm before giving the final answer." The result of 25 * 4 = 100, which is greater than 50, so I asked for confirmation before providing the final answer.\n\nNow the user has given me permission to provide the result, so I should tell them:\n1. Why I stopped (the rule about results > 50)\n2. The result, which is 100', signature='EocGCmQIDBgCKkAFHFiorjrHqbSqhgMFHBjVqjBmknbL/nQf1WyDwPEDP1j0qNVPljdBd6Ae/1/98ri3KltMbKaogtruEXg8Opc1MhpjbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOTgAEgyagPbCRb1fEIBYtcoaDJsg5DmfCeRrO0cLNSIwKx

In [15]:
for msg in agent.agent_config.context_messages:
    print(msg.role)
    for content_block in msg.content:
        if content_block.content_block_type == "text":
            content = content_block.text
        elif content_block.content_block_type == "thinking":
            content = content_block.thinking
        else:
            content = content_block.to_dict()
        print(content_block.content_block_type, content)
    print("\n---\n")

Role.USER
ContentBlockType.TEXT Calculate 25 * 4 for me.

---

Role.ASSISTANT
ContentBlockType.THINKING The user wants me to calculate 25 * 4. I should use the quick_calculate function for this.

Let me calculate: 25 * 4 = 100

Since the result is 100, which is greater than 50, I need to ask for confirmation before giving the final answer.

Let me first calculate it, then check if I need confirmation.
ContentBlockType.TOOL_USE {'content_block_type': 'tool_use', 'tool_name': 'quick_calculate', 'tool_id': 'toolu_01VsgsnshqESPxysS8NADFgt', 'tool_input': {'expression': '25 * 4'}, 'kwargs': {'caller': {'type': 'direct'}}}

---

Role.USER
ContentBlockType.TOOL_RESULT {'content_block_type': 'tool_result', 'tool_name': 'quick_calculate', 'tool_id': 'toolu_01VsgsnshqESPxysS8NADFgt', 'tool_result': [{'content_block_type': 'text', 'text': '100', 'kwargs': {}}], 'is_error': False, 'kwargs': {}}

---

Role.ASSISTANT
ContentBlockType.TEXT Since the result is 100, which is greater than 50, I need to 